In [0]:
import os
import re
import json
import cv2
import pytesseract
import pandas as pd
import logging
import matplotlib.pyplot as plt
from PIL import Image


# =============================================================================
# Logging Configuration & Utilities
# =============================================================================
class ExcludeLogsFilter(logging.Filter):
    """
    Custom filter to remove log messages containing unwanted substrings.
    This filter prevents cluttering the logs with low-value debug messages in production.
    """
    def filter(self, record):
        exclude_keywords = [
            "spark.databricks.clusterUsageTags.sparkVersion",
            "Answer received",
            "Command to send"
        ]
        return not any(kw in record.getMessage() for kw in exclude_keywords)


def configure_logger(debug: bool = False, log_file: str = "logs/application.log") -> logging.Logger:
    """
    Configures a logger with a file handler and a stream handler.
    The debug flag controls the level of detail in the logs.
    """
    os.makedirs(os.path.dirname(log_file), exist_ok=True)
    logger = logging.getLogger("PipelineLogger")
    logger.setLevel(logging.DEBUG if debug else logging.INFO)

    # Remove previous handlers
    for h in logger.handlers[:]:
        logger.removeHandler(h)

    formatter = logging.Formatter("%(asctime)s %(levelname)-8s %(message)s")

    # File Handler
    fh = logging.FileHandler(log_file)
    fh.setFormatter(formatter)
    fh.addFilter(ExcludeLogsFilter())
    logger.addHandler(fh)

    # Stream Handler
    ch = logging.StreamHandler()
    ch.setFormatter(formatter)
    ch.addFilter(ExcludeLogsFilter())
    logger.addHandler(ch)

    return logger


# Global logger instance; will be (re)configured in main()
logger = configure_logger(debug=False)


# =============================================================================
# Shared Utility Classes
# =============================================================================
class FileUtils:
    """Utilities to work with file paths in DBFS and local file systems."""
    @staticmethod
    def dbfs_to_local_path(dbfs_path: str) -> str:
        if dbfs_path.startswith("dbfs:/"):
            local_path = os.path.join("/dbfs", dbfs_path[len("dbfs:/"):].lstrip("/"))
            logger.debug(f"Converted '{dbfs_path}' to local path '{local_path}'.")
            return local_path
        return dbfs_path

    @staticmethod
    def sanitize_section_name(section: str) -> str:
        safe_name = section.lower().replace(" ", "_").replace("/", "_").replace(":", "")
        logger.debug(f"Sanitized section name '{section}' to '{safe_name}'.")
        return safe_name


class ImageUtils:
    """Image loading, preprocessing and OCR helper methods."""
    @staticmethod
    def safe_read_image(image_path: str):
        local_path = FileUtils.dbfs_to_local_path(image_path)
        logger.info(f"Reading image (OpenCV) from: {local_path}")
        if not os.path.exists(local_path):
            logger.error(f"File not found at {local_path}")
            raise FileNotFoundError(f"File not found at {local_path}")
        img = cv2.imread(local_path)
        if img is None:
            logger.error(f"OpenCV failed to read image at {local_path}")
            raise ValueError(f"OpenCV failed to read image at {local_path}")
        return img

    @staticmethod
    def safe_read_image_pil(image_path: str) -> Image.Image:
        local_path = FileUtils.dbfs_to_local_path(image_path)
        logger.info(f"Reading image (PIL) from: {local_path}")
        if not os.path.exists(local_path):
            logger.error(f"File not found at {local_path}")
            raise FileNotFoundError(f"File not found at {local_path}")
        return Image.open(local_path)

    @staticmethod
    def preprocess_image(img, debug: bool = False):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 15, 9
        )
        # if debug:
        #     plt.figure(figsize=(8, 8))
        #     plt.imshow(thresh, cmap="gray")
        #     plt.title("Thresholded Image")
        #     plt.axis("off")
        #     plt.show()
        return thresh

    @staticmethod
    def detect_text_regions(thresh_img, debug: bool = False):
        contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rois = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 30 and h > 15:
                rois.append((x, y, w, h))
        rois.sort(key=lambda b: (b[1], b[0]))
        if debug:
            debug_img = cv2.cvtColor(thresh_img, cv2.COLOR_GRAY2BGR)
            for (x, y, w, h) in rois:
                cv2.rectangle(debug_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
            plt.figure(figsize=(10, 10))
            plt.imshow(debug_img)
            plt.title("Detected Text Regions")
            plt.axis("off")
            plt.show()
        return rois

    @staticmethod
    def perform_ocr_on_rois(img, rois, debug: bool = False):
        results = []
        for (x, y, w, h) in rois:
            roi = img[y:y+h, x:x+w]
            text = pytesseract.image_to_string(roi, config="--psm 6").strip() or "[BLANK]"
            results.append((x, y, w, h, text))
            if debug:
                logger.debug(f"OCR result at ({x}, {y}, {w}, {h}): {text}")
        return results


def group_ocr_rows(roi_results, y_threshold=20):
    """
    Group OCR result bounding boxes by their y-coordinate (rows).
    Each element is expected to be a tuple: (x, y, w, h, text).
    """
    rois_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
    rois_with_center.sort(key=lambda r: r[5])
    groups = []
    current_group = []
    current_center = None

    for (x, y, w, h, text, center) in rois_with_center:
        if current_center is None:
            current_group.append((x, y, w, h, text))
            current_center = center
        elif abs(center - current_center) <= y_threshold:
            current_group.append((x, y, w, h, text))
            current_center = (current_center + center) / 2
        else:
            groups.append(current_group)
            current_group = [(x, y, w, h, text)]
            current_center = center
    if current_group:
        groups.append(current_group)
    return groups


In [0]:

# =============================================================================
# Pipeline Classes
# =============================================================================
class DailyDrillingReportPipeline:
    """Processes the Daily Drilling Report section."""
    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        # Crop region based on known coordinates (modify as needed)
        x, y, w, h = 1600, 0, 950, 185
        cropped = img[y:y+h, x:x+w]
        gray = cropped if len(cropped.shape) == 2 else cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
        equalized = cv2.equalizeHist(gray)
        blurred = cv2.GaussianBlur(equalized, (5, 5), 0)
        processed = cv2.adaptiveThreshold(blurred, 255,
                                          cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                          cv2.THRESH_BINARY, 11, 2)
        ocr_text = pytesseract.image_to_string(processed, config="--psm 6").strip()
        logger.info("Daily Drilling Report OCR extraction complete.")

        # Extract expected keys (customize the regex as needed)
        expected_keys = ["Report Date", "Report Num", "Rig"]
        combined = " ".join(ocr_text.splitlines()).strip()
        combined = re.sub(r'\s+', ' ', combined)
        extracted = {}
        for i, key in enumerate(expected_keys):
            if i < len(expected_keys) - 1:
                pattern = rf'{re.escape(key)}\s*:\s*(.*?)(?=\s*{re.escape(expected_keys[i+1])}\s*:|$)'
            else:
                pattern = rf'{re.escape(key)}\s*:\s*(.*)'
            match = re.search(pattern, combined, re.IGNORECASE)
            extracted[key] = match.group(1).strip() if (match and match.group(1).strip()) else None

        df = pd.DataFrame(list(extracted.items()), columns=["Key", "Value"])
        return {"DAILY DRILLING REPORT": extracted}, df

class WellJobInfoPipeline:
    """Processes the Well/Job information section."""
    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        ocr_text = pytesseract.image_to_string(img, config="--psm 6").strip()
        expected_keys = [
            "Well Name", "Job Name", "Supervisor(s)", "Field", "Sec/Twn/Rng",
            "Phone", "AFE #", "API #", "Email", "Contractor", "Elevation",
            "RKB", "Spud Date", "Days from Spud", "Days on Loc", "MD/TVD",
            "24 Hr Footage", "Present Operations", "Activity Planned"
        ]
        combined = " ".join(ocr_text.splitlines()).strip()
        combined = re.sub(r'\s+', ' ', combined)
        result = {}
        for i, key in enumerate(expected_keys):
            if i < len(expected_keys) - 1:
                pattern = rf'{re.escape(key)}\s*:\s*(.*?)(?=\s*{re.escape(expected_keys[i+1])}\s*:|$)'
            else:
                pattern = rf'{re.escape(key)}\s*:\s*(.*)'
            match = re.search(pattern, combined, re.IGNORECASE)
            result[key] = match.group(1).strip() if match else ""
        df = pd.DataFrame(list(result.items()), columns=["Key", "Value"])
        logger.info("Well/Job Information processed.")
        return {"WELL/JOB INFORMATION": result}, df


class MudPipeline:
    """Processes the Mud section."""
    @staticmethod
    def read_cropped_section_image(image_path: str):
        return ImageUtils.safe_read_image(image_path)

    @staticmethod
    def preprocess_image_mud(img, debug: bool = False):
        return ImageUtils.preprocess_image(img, debug=debug)

    @staticmethod
    def detect_text_regions_mud(thresh_img, debug: bool = False):
        return ImageUtils.detect_text_regions(thresh_img, debug=debug)

    @staticmethod
    def perform_ocr_on_rois_mud(img, rois, debug: bool = False):
        return ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)

    @staticmethod
    def parse_value_row_tokens(expected_headers, tokens):
        expected_token_count = (len(expected_headers) - 1) + 3
        if len(tokens) < expected_token_count:
            tokens += [""] * (expected_token_count - len(tokens))
        elif len(tokens) > expected_token_count:
            tokens = tokens[:expected_token_count]
        result = {}
        idx = 0
        for header in expected_headers:
            if header == "GELS (10s/10m/30m)":
                gels_tokens = tokens[idx:idx+3]
                result[header] = {"10s": gels_tokens[0], "10m": gels_tokens[1], "30m": gels_tokens[2]}
                idx += 3
            else:
                result[header] = tokens[idx]
                idx += 1
        return result

    @staticmethod
    def build_mud_dict_from_rois(roi_texts, expected_headers):
        row_tolerance = 10
        rows = []
        current_row = []
        prev_y = None
        for (x, y, w, h, text) in roi_texts:
            if prev_y is None or abs(y - prev_y) <= row_tolerance:
                current_row.append((x, y, w, h, text))
            else:
                rows.append(current_row)
                current_row = [(x, y, w, h, text)]
            prev_y = y
        if current_row:
            rows.append(current_row)

        row_strings = []
        for row in rows:
            row.sort(key=lambda cell: cell[0])
            line = " ".join(cell[4] for cell in row)
            row_strings.append(line)

        # Identify header and value rows (customize rules as needed)
        header1_line = None
        value1_line = None
        header2_line = None
        value2_line = None
        for i, r_text in enumerate(row_strings):
            if "type" in r_text.lower() and not header1_line:
                header1_line = r_text
                if i+1 < len(row_strings):
                    value1_line = row_strings[i+1]
            elif header1_line and not header2_line and any(kw in r_text.lower() for kw in ["rpm", "mud", "loss", "comments"]):
                header2_line = r_text
                if i+1 < len(row_strings):
                    value2_line = row_strings[i+1]
                break

        if value1_line is None:
            logger.error("No data row found for Mud section!")
            return {}
        tokens1 = value1_line.split()
        tokens2 = value2_line.split() if value2_line else []
        combined_tokens = tokens1 + tokens2
        return MudPipeline.parse_value_row_tokens(expected_headers, combined_tokens)

    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = MudPipeline.read_cropped_section_image(image_path)
        thresh_img = MudPipeline.preprocess_image_mud(img, debug=debug)
        rois = MudPipeline.detect_text_regions_mud(thresh_img, debug=debug)
        roi_texts = MudPipeline.perform_ocr_on_rois_mud(img, rois, debug=debug)
        expected_headers = [
            "Type", "Weight In", "Weight Out", "pH", "CAKE",
            "GELS (10s/10m/30m)", "Oil/Water", "FV", "ES", "PV",
            "YP", "CL", "Ca", "LGS", "WL", "HTHP Loss", "3 RPM",
            "6 RPM", "Mud Pits and Hole Volume", "24 Hr Loss",
            "Total Loss", "Comments"
        ]
        mud_dict = MudPipeline.build_mud_dict_from_rois(roi_texts, expected_headers)
        if isinstance(mud_dict, dict):
            df = pd.DataFrame(list(mud_dict.items()), columns=["Key", "Value"])
        else:
            df = pd.DataFrame(mud_dict)
        # logger.info("===== FINAL MUD DATA =====")
        # logger.info(json.dumps({"MUD": mud_dict}, indent=4))
        return {"MUD": mud_dict}, df

class SurveyDataPipeline:
    """Processes the Survey Data section."""
    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 15, 9)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rois = [(x, y, w, h) for cnt in contours for (x, y, w, h) in [cv2.boundingRect(cnt)]
                if w > 30 and h > 15]
        rois.sort(key=lambda b: (b[1], b[0]))
        roi_texts = []
        for (x, y, w, h) in rois:
            roi = img[y:y+h, x:x+w]
            text = pytesseract.image_to_string(roi, config="--psm 6").strip() or ""
            roi_texts.append((x, y, w, h, text))
        groups = group_ocr_rows(roi_texts, y_threshold=10)
        survey_list = []
        for group in groups:
            group.sort(key=lambda cell: cell[0])
            tokens = [cell[4] for cell in group]
            if len(tokens) >= 5:
                row_dict = {"MD": tokens[0], "Inclination": tokens[1],
                            "Azimuth": tokens[2], "DLS": tokens[3], "TVD": tokens[4]}
                survey_list.append(row_dict)
        df = pd.DataFrame(survey_list)
        logger.info("Survey Data processed.")
        return {"SURVEY DATA": survey_list}, df

class CostDataPipeline:
    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ocr_text = pytesseract.image_to_string(gray, config="--psm 6")
        logger.info("Cost OCR extraction complete.")
        expected_keys = [
            "Drilling AFE Amount", "Daily Drilling Cost", "Cumulative Drilling Cost",
            "Cumulative Well Cost", "Daily Mud Cost", "Cumulative Mud Cost"
        ]
        combined = " ".join(ocr_text.splitlines()).strip()
        combined = re.sub(r'\s+', ' ', combined)
        extracted = {}
        for i, key in enumerate(expected_keys):
            if i < len(expected_keys) - 1:
                pattern = rf'{re.escape(key)}\s*:\s*(.*?)(?=\s*{re.escape(expected_keys[i+1])}\s*:|$)'
            else:
                pattern = rf'{re.escape(key)}\s*:\s*(.*)'
            match = re.search(pattern, combined, re.IGNORECASE)
            extracted[key] = match.group(1).strip() if match and match.group(1).strip() else "[Blank]"
        df = pd.DataFrame(list(extracted.items()), columns=["Key", "Value"])
        logger.info(f"COST DataFrame shape: {df.shape}")
        return {"COST DATA": extracted}, df


In [0]:

class ObsIntPipeline:
    @staticmethod
    def build_obs_int_data(roi_texts):
        header_tokens = {"daily numbers: observation & intervention", "number"}
        tokens = []
        for (_, _, _, _, text) in roi_texts:
            for line in re.split(r'\n+', text):
                token = line.strip()
                if token and token.lower() not in header_tokens:
                    tokens.append(token)
        logger.debug(f"Consolidated OCR tokens: {tokens}")
        expected_types = ["Stop Cards", "Hazard ID's", "JSA's", "Permit to Work", "Totals"]
        types_found = []
        remaining_tokens = tokens.copy()
        for etype in expected_types:
            found = None
            for t in tokens:
                if etype.lower() in t.lower():
                    found = etype
                    if t in remaining_tokens:
                        remaining_tokens.remove(t)
                    break
            types_found.append(etype)
        logger.debug(f"Identified types: {types_found}")
        numbers = [t if re.match(r'^\d+(\.\d+)?$', t) else "" for t in remaining_tokens]
        logger.debug(f"Initial numeric tokens: {numbers}")
        if len(numbers) > len(expected_types) and numbers[0] == "":
            numbers = numbers[1:]
        while len(numbers) < len(expected_types):
            numbers.append("")
        if len(numbers) > len(expected_types):
            numbers = numbers[:len(expected_types)]
        logger.debug(f"Final numeric tokens: {numbers}")
        records = []
        for t, n in zip(expected_types, numbers):
            records.append({"Type": t, "Number": n})
        logger.info(f"Final Obs & Int structured data: {records}")
        df = pd.DataFrame(records)
        return records, df

    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        records, df = ObsIntPipeline.build_obs_int_data(roi_texts)
        return {"DAILY NUMBERS: OBSERVATION & INTERVENTION": records}, df
 
 
class PersonnelPipeline:
    @staticmethod
    def preprocess_personnel_data(groups):
        personnel_data = []
        # Define header lines to skip (in lowercase)
        header_lines = {"personnel", "company contractor no. personnel daily hours cumulative hours", "ssn"}
        for group in groups:
            row_text = group.strip()
            if row_text.lower() in header_lines:
                continue
            tokens = row_text.split()
            # Company: tokens before first convertible-to-number token.
            company_tokens = []
            for token in tokens:
                try:
                    float(token)
                    break
                except ValueError:
                    company_tokens.append(token)
            company = " ".join(company_tokens)
            numeric_tokens = re.findall(r'\d+(?:\.\d+)?', row_text)
            if len(numeric_tokens) < 3:
                continue
            try:
                no_personnel = int(float(numeric_tokens[-3]))
                daily_hours = int(float(numeric_tokens[-2]))
                cumulative_hours = int(float(numeric_tokens[-1]))
            except Exception:
                continue
            personnel_data.append({
                "Company": company,
                "Contractor": "Service Company",
                "No. Personnel": no_personnel,
                "Daily Hours": daily_hours,
                "Cumulative Hours": cumulative_hours
            })
        return {"PERSONNEL": personnel_data}

    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = safe_read_image(image_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 11, 2)
        rois = detect_text_regions(thresh, debug=debug)
        roi_results = perform_ocr_on_rois(img, rois, debug=debug)
        groups = group_ocr_rows(roi_results, y_threshold=20)
        data_dict = PersonnelPipeline.preprocess_personnel_data(groups)
        if not data_dict.get("PERSONNEL"):
            df = pd.DataFrame(columns=["Company", "Contractor", "No. Personnel", "Daily Hours", "Cumulative Hours"])
        else:
            df = pd.DataFrame(data_dict["PERSONNEL"])
        return data_dict, df

class BHAPipeline:
    @staticmethod
    def extract_bha_data(image_path: str):
        image = ImageUtils.safe_read_image_pil(image_path)
        ocr_text = pytesseract.image_to_string(image)
        patterns = {
            "Drill Pipe Detail": r"Drill Pipe Detail:\s*([^\n]+)",
            "Size": r"Size:\s*([\d.]+)\b",
            "Wt./Ft": r"Wt\./Ft:\s*([\d.]+)\b",
            "Connection": r"Connection:\s*([\w\d-]+)\b",
            "ID": r"ID:\s*([\d.]+)\b",
            "Drill Bit": r"Drill Bit:\s*([^\n;]+)",
            "Motor": r"Motor:\s*([^\n;]+)",
            "MWD Tool": r"MWD Tool:\s*([^\n;]+)",
            "Monel Collar": r"Monel Collar:\s*([^\n;]+)",
            "X-Over": r"X-Over:\s*([^\n;]+)",
            "Sub": r"Sub:\s*([^\n;]+)",
            "HWDP": r"HWDP:\s*([^\n;]+)",
            "Drill Pipe": r"Drill Pipe:\s*([\d.]+(?:\" DP)?)",
            "Reamer": r"Reamer:\s*([^\n;]+)",
            "Shock Sub": r"Shock Sub:\s*([^\n;]+)",
            "Total Length": r"Total Length:\s*(\d+)\b"
        }
        bha_data = {}
        for key, pat in patterns.items():
            match = re.search(pat, ocr_text)
            if match:
                bha_data[key] = match.group(1).strip()
        if "Drill Pipe Detail" in bha_data:
            detail = bha_data["Drill Pipe Detail"]
            for rem in ["Size", "Wt./Ft", "Connection", "ID"]:
                if rem in bha_data:
                    detail = re.sub(rf"{rem}:\s*{re.escape(bha_data[rem])}", "", detail).strip(",; ")
            bha_data["Drill Pipe Detail"] = detail
        structured = {
            "BHA": {
                "Drill Pipe Detail": bha_data.get("Drill Pipe Detail", ""),
                "Size": bha_data.get("Size", ""),
                "Wt./Ft": bha_data.get("Wt./Ft", ""),
                "Connection": bha_data.get("Connection", ""),
                "ID": bha_data.get("ID", ""),
                "BHA #4": {
                    "Drill Bit": bha_data.get("Drill Bit", ""),
                    "Motor": bha_data.get("Motor", ""),
                    "MWD Tool": bha_data.get("MWD Tool", ""),
                    "Monel Collar": bha_data.get("Monel Collar", ""),
                    "X-Over": bha_data.get("X-Over", ""),
                    "Sub": bha_data.get("Sub", ""),
                    "HWDP": bha_data.get("HWDP", ""),
                    "Drill Pipe": bha_data.get("Drill Pipe", ""),
                    "Reamer": bha_data.get("Reamer", ""),
                    "Shock Sub": bha_data.get("Shock Sub", "")
                },
                "Total Length": bha_data.get("Total Length", "")
            }
        }
        return structured

    @staticmethod
    def process(image_path: str, debug: bool = False):
        bha_json = BHAPipeline.extract_bha_data(image_path)
        df = pd.json_normalize(bha_json["BHA"])
        return {"BHA": bha_json["BHA"]}, df

class DirInfoPipeline:
    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        return DirInfoPipeline.build_dir_info(roi_texts, debug=debug)

    @staticmethod
    def build_dir_info(roi_texts, debug: bool = False):
        all_texts = [t[4] for t in roi_texts]
        daily_cum_idx = next((i for i, txt in enumerate(all_texts)
                              if "daily" in txt.lower() and "cumulative" in txt.lower()), None)
        if daily_cum_idx is None:
            logger.warning("Could not find 'Daily Cumulative' box.")
            return {}, pd.DataFrame()
        cat_idx = daily_cum_idx + 1
        if cat_idx >= len(all_texts):
            logger.warning("No bounding box after 'Daily Cumulative'.")
            return {}, pd.DataFrame()
        categories_box = all_texts[cat_idx]
        lines = [ln.strip() for ln in categories_box.split("\n") if ln.strip()]
        if len(lines) < 5:
            logger.warning(f"Expected at least 5 category lines, got {len(lines)}: {lines}")
        def safe_get(idx):
            return all_texts[idx] if 0 <= idx < len(all_texts) else ""
        structured = []
        for i in range(4):
            cat_name = lines[i] if i < len(lines) else f"Unknown Category {i+1}"
            daily_box = safe_get(cat_idx + 1 + (i * 2))
            cum_box = safe_get(cat_idx + 2 + (i * 2))
            structured.append({
                "Category": cat_name,
                "Daily": daily_box if daily_box else "",
                "Cumulative": cum_box if cum_box else ""
            })
        last_box = safe_get(cat_idx + 9)
        last_cat = lines[4] if len(lines) >= 5 else "Rotating Footage"
        remainder = last_box.replace(last_cat, "").strip() if last_box else ""
        tokens = remainder.split()
        daily_val = tokens[0] if len(tokens) >= 2 else ""
        cum_val = tokens[1] if len(tokens) >= 2 else ""
        structured.append({
            "Category": last_cat,
            "Daily": daily_val,
            "Cumulative": cum_val if cum_val != "]" else ""
        })
        df = pd.DataFrame(structured)
        logger.info(f"DIR INFO DataFrame shape: {df.shape}")
        return {"DIR INFO": structured}, df


In [0]:
# =======================
# Drill Bits Pipeline
# =======================
class DrillBitsPipeline:
    @staticmethod
    def build_drill_bits_info(roi_texts, debug: bool = False):
        row_tolerance = 10
        grouped_rows = []
        current_row = []
        prev_y = None
        for (x, y, w, h, text) in roi_texts:
            if prev_y is None or abs(y - prev_y) <= row_tolerance:
                current_row.append((x, y, w, h, text))
            else:
                grouped_rows.append(current_row)
                current_row = [(x, y, w, h, text)]
            prev_y = y
        if current_row:
            grouped_rows.append(current_row)
        row_strings = []
        for i, row in enumerate(grouped_rows):
            row.sort(key=lambda cell: cell[0])
            line = " ".join(cell[4] for cell in row).replace("\n", " ").strip()
            row_strings.append(line)
            logger.info(f"Drill Bits Row {i}: {line}")
        if len(row_strings) < 3:
            logger.warning("Not enough rows for Drill Bits layout.")
            return []
        data_lines = row_strings[3:]
        final_columns = [
            "Bit #", "Size", "Make", "Model", "Serial #",
            "Nozzle-(Number x Size)", "Nozzle-TFA",
            "Depth-In", "Depth-Out", "Depth-Feet", "Depth-ROP",
            "Hours-Total", "Hours-On Btm",
            "Dull Grade-I", "Dull Grade-O1", "Dull Grade-D", "Dull Grade-L", 
            "Dull Grade-B", "Dull Grade-G", "Dull Grade-O2", "Dull Grade-RP"
        ]
        structured_data = []
        for line in data_lines:
            tokens = line.split()
            if len(tokens) < len(final_columns):
                tokens += [""] * (len(final_columns) - len(tokens))
            elif len(tokens) > len(final_columns):
                tokens = tokens[:len(final_columns)]
            row_dict = {final_columns[i]: tokens[i] for i in range(len(final_columns))}
            structured_data.append(row_dict)
            logger.info(f"Drill Bits Parsed row: {row_dict}")
        return structured_data

    @staticmethod
    def process(image_path: str, debug: bool = False):
        # Use ImageUtils instead of undefined ImageUtils
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        drill_bits = DrillBitsPipeline.build_drill_bits_info(roi_texts, debug=debug)
        # Wrap output in a dict so that aggregator code can call .get()
        return {"DRILL BITS": drill_bits}, None
        
class ConsumablesPipeline:
    @staticmethod
    def detect_text_regions_consumables(thresh_img, debug: bool = False):
        contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rois = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 30 and h > 15:
                rois.append((x, y, w, h))
        rois.sort(key=lambda b: (b[1], b[0]))
        return rois

    @staticmethod
    def group_rois_by_row(roi_results, threshold: int = 20):
        roi_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
        roi_with_center.sort(key=lambda r: r[5])
        groups = []
        current_group = []
        current_center = None
        for (x, y, w, h, text, y_center) in roi_with_center:
            if current_center is None:
                current_group.append(text)
                current_center = y_center
            elif abs(y_center - current_center) < threshold:
                current_group.append(text)
            else:
                groups.append(" ".join(current_group))
                current_group = [text]
                current_center = y_center
        if current_group:
            groups.append(" ".join(current_group))
        return groups

    @staticmethod
    def build_consumables_dict_from_rois(roi_texts, debug: bool = False):
        groups = ConsumablesPipeline.group_rois_by_row(roi_texts, threshold=20)
        data_rows = []
        for line in groups:
            line_str = line.strip()
            if ("consumable" in line_str.lower() and "received" in line_str.lower()) or "nun" in line_str.lower():
                continue
            if len(line_str.split()) < 5:
                continue
            data_rows.append(line_str)
        consumables_list = []
        for line in data_rows:
            tokens = re.split(r'\s+', line)
            if len(tokens) > 5:
                first = " ".join(tokens[:-4])
                tokens = [first] + tokens[-4:]
            if len(tokens) != 5:
                continue
            consumables_list.append({
                "Consumable": tokens[0],
                "Daily Received (gal)": tokens[1],
                "Daily Used (gal)": tokens[2],
                "Cumulative Used (gal)": tokens[3],
                "Daily on Hand (gal)": tokens[4]
            })
        return consumables_list

    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ConsumablesPipeline.detect_text_regions_consumables(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        consumables_list = ConsumablesPipeline.build_consumables_dict_from_rois(roi_texts, debug=debug)
        df = pd.DataFrame(consumables_list)
        return {"CONSUMABLES": consumables_list}, df
    
# # =============================================================================
# # CASING Pipeline
# # =============================================================================
class CasingPipeline:
    @staticmethod
    def build_casing_dict_from_rois(roi_texts, expected_headers, debug=False):
        # Group OCR results into rows using the common grouping function.
        grouped_rows = group_ocr_rows(roi_texts, y_threshold=20)
        logger.debug(f"Grouped into {len(grouped_rows)} row groups.")
        casing_rows = []
        for group in grouped_rows:
            # Assemble a single row string; sort by x coordinate.
            row_string = " ".join([text for (x, y, w, h, text) in sorted(group, key=lambda item: item[0])]).strip()
            # Skip header rows (assumed to include both "type" and "size")
            if "type" in row_string.lower() and "size" in row_string.lower():
                if debug:
                    logger.info(f"Skipping header row: {row_string}")
                continue
            tokens = re.split(r'\s{2,}', row_string)
            if len(tokens) == 1:
                tokens = row_string.split()
            if len(tokens) < len(expected_headers):
                logger.warning(f"Row skipped due to insufficient tokens: {tokens}")
                continue
            tokens = tokens[:len(expected_headers)]
            row_dict = {expected_headers[i]: tokens[i] for i in range(len(expected_headers))}
            casing_rows.append(row_dict)
            if debug:
                logger.info(f"Processed CASING row: {row_dict}")
        return casing_rows

    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        expected_headers = ["Type", "Size", "Weight", "Grade", "Connection", "Top MD", "Bottom MD", "TOC"]
        casing_data = CasingPipeline.build_casing_dict_from_rois(roi_texts, expected_headers, debug=debug)
        return {"CASING": casing_data}, pd.DataFrame(casing_data)


In [0]:

# import os
# import re
# import json
# import cv2
# import pytesseract
# import pandas as pd
# import logging
# from PIL import Image

# # =============================================================================
# # Logging Configuration (Assumed to be already set up)
# # =============================================================================
# def configure_logger(debug: bool = False, log_file: str = "logs/application.log") -> logging.Logger:
#     os.makedirs(os.path.dirname(log_file), exist_ok=True)
#     logger = logging.getLogger("PipelineLogger")
#     logger.setLevel(logging.DEBUG if debug else logging.INFO)
#     for h in logger.handlers[:]:
#         logger.removeHandler(h)
#     formatter = logging.Formatter("%(asctime)s %(levelname)-8s %(message)s")
#     fh = logging.FileHandler(log_file)
#     fh.setFormatter(formatter)
#     logger.addHandler(fh)
#     ch = logging.StreamHandler()
#     ch.setFormatter(formatter)
#     logger.addHandler(ch)
#     return logger

# logger = configure_logger(debug=True)

# # =============================================================================
# # Shared Utilities (File and Image Helpers)
# # =============================================================================
# class FileUtils:
#     @staticmethod
#     def dbfs_to_local_path(dbfs_path: str) -> str:
#         if dbfs_path.startswith("dbfs:/"):
#             local_path = os.path.join("/dbfs", dbfs_path[len("dbfs:/"):].lstrip("/"))
#             logger.debug(f"Converted '{dbfs_path}' to local path '{local_path}'.")
#             return local_path
#         return dbfs_path

#     @staticmethod
#     def sanitize_section_name(section: str) -> str:
#         safe_name = section.lower().replace(" ", "_").replace("/", "_").replace(":", "")
#         logger.debug(f"Sanitized section name '{section}' to '{safe_name}'.")
#         return safe_name

# class ImageUtils:
#     @staticmethod
#     def safe_read_image(image_path: str):
#         local_path = FileUtils.dbfs_to_local_path(image_path)
#         logger.info(f"Reading image using OpenCV from: {local_path}")
#         if not os.path.exists(local_path):
#             logger.error(f"File not found at {local_path}")
#             raise FileNotFoundError(f"File not found at {local_path}")
#         img = cv2.imread(local_path)
#         if img is None:
#             logger.error(f"OpenCV failed to read image at {local_path}")
#             raise ValueError(f"OpenCV failed to read image at {local_path}")
#         return img

#     @staticmethod
#     def safe_read_image_pil(image_path: str) -> Image.Image:
#         local_path = FileUtils.dbfs_to_local_path(image_path)
#         logger.info(f"Reading image using PIL from: {local_path}")
#         if not os.path.exists(local_path):
#             logger.error(f"File not found at {local_path}")
#             raise FileNotFoundError(f"File not found at {local_path}")
#         return Image.open(local_path)

#     @staticmethod
#     def preprocess_image(img, debug: bool = False):
#         gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#         thresh = cv2.adaptiveThreshold(
#             gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#             cv2.THRESH_BINARY, 15, 9
#         )
#         return thresh

#     @staticmethod
#     def detect_text_regions(thresh_img, debug: bool = False):
#         contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#         rois = []
#         for cnt in contours:
#             x, y, w, h = cv2.boundingRect(cnt)
#             if w > 30 and h > 15:
#                 rois.append((x, y, w, h))
#         rois.sort(key=lambda b: (b[1], b[0]))
#         return rois

#     @staticmethod
#     def perform_ocr(image, config="--psm 6") -> str:
#         text = pytesseract.image_to_string(image, config=config)
#         return text.strip()

#     @staticmethod
#     def perform_ocr_on_rois(img, rois, debug: bool = False):
#         results = []
#         for (x, y, w, h) in rois:
#             roi = img[y:y+h, x:x+w]
#             text = pytesseract.image_to_string(roi, config="--psm 6").strip() or "[BLANK]"
#             results.append((x, y, w, h, text))
#             if debug:
#                 logger.debug(f"OCR extracted from ({x}, {y}, {w}, {h}): {text}")
#         return results

# def group_ocr_rows(roi_results, y_threshold=20):
#     rois_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
#     rois_with_center.sort(key=lambda r: r[5])
#     groups = []
#     current_group = []
#     current_center = None
#     for (x, y, w, h, text, center) in rois_with_center:
#         if current_center is None:
#             current_group.append((x, y, w, h, text))
#             current_center = center
#         elif abs(center - current_center) <= y_threshold:
#             current_group.append((x, y, w, h, text))
#             current_center = (current_center + center) / 2
#         else:
#             groups.append(current_group)
#             current_group = [(x, y, w, h, text)]
#             current_center = center
#     if current_group:
#         groups.append(current_group)
#     return groups

# # # For convenience, let safe_read_image be the OpenCV version.
# # safe_read_image = ImageUtils.safe_read_image

# # =============================================================================
# # Pumps Pipeline
# # =============================================================================
# def parse_pumps_table(ocr_text: str) -> list:
#     pump_pattern = re.compile(
#         r"^(\d+)?\s*"               # Number (optional)
#         r"(BOMCO)\s+(TRIPLEX)\s+"    # Model, Type
#         r"(\d+)?\s*"                # HHP (optional)
#         r"(\d+)\s+"                 # Efficiency
#         r"([\d.]+)\s+"              # Stroke(in)
#         r"([\d.]+)\s+"              # Liner(in)
#         r"(\d+)\s+"                 # P-Rating(psi)
#         r"(\d+)\s+"                 # P-Limit(psi)
#         r"(\d+)\s+"                 # SPM Rating
#         r"(\d+)\s*$",               # SPM Limit
#         re.IGNORECASE
#     )
#     pumps = []
#     for line in ocr_text.splitlines():
#         line = line.strip()
#         match = pump_pattern.match(line)
#         if match:
#             (number, model, pump_type, hhp, efficiency, stroke,
#              liner, p_rating, p_limit, spm_rating, spm_limit) = match.groups()
#             pumps.append({
#                 "Number": number if number else "",
#                 "Model": model,
#                 "Type": pump_type,
#                 "HHP": hhp if hhp else "",
#                 "Efficiency": efficiency,
#                 "Stroke(in)": stroke,
#                 "Liner(in)": liner,
#                 "P-Rating(psi)": p_rating,
#                 "P-Limit(psi)": p_limit,
#                 "SPM Rating": spm_rating,
#                 "SPM Limit": spm_limit
#             })
#     return pumps

# def parse_drilling_circ_rates(ocr_text: str) -> list:
#     circ_rates = []
#     segments = re.split(r"(?=Drilling/Circ Rate \d+)", ocr_text)
#     pattern = re.compile(
#         r"Drilling/Circ Rate\s+(\d+).*?"       # Rate ID
#         r"(\d+)\s+PS[!I].*?"                   # Pressure
#         r"@\s*(\d+).*?"                        # SPM value
#         r"([\d.]+)\s+Gal/Stoke.*?"              # Gal/Stoke
#         r"([\d.]+)\s+GPM.*?"                    # GPM
#         r"([\d.]+)\s+BPM.*?"                    # BPM
#         r"([\d.]+)\s+DC.*?"                     # DC
#         r"([\d.]+)\s+DP",                      # DP
#         re.IGNORECASE | re.DOTALL
#     )
#     for seg in segments:
#         seg = seg.strip()
#         if not seg.startswith("Drilling/Circ Rate"):
#             continue
#         seg_clean = " ".join(seg.splitlines())
#         match = pattern.search(seg_clean)
#         if match:
#             rate_id, pressure, spm, gal_stroke, gpm, bpm, dc, dp = match.groups()
#             circ_rates.append({
#                 "RateID": rate_id,
#                 "Pressure(PSI)": pressure,
#                 "SPM": spm,
#                 "Gal/Stoke": gal_stroke,
#                 "GPM": gpm,
#                 "BPM": bpm,
#                 "DC": dc,
#                 "DP": dp
#             })
#         else:
#             logger.warning(f"PumpsPipeline: No match found in segment:\n{seg_clean}")
#     return circ_rates

# class PumpsPipeline:
#     @staticmethod
#     def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
#         # Using PIL for OCR
#         pil_img = ImageUtils.safe_read_image_pil(image_path)
#         ocr_text = ImageUtils.perform_ocr(pil_img, config="--psm 6")
#         if debug:
#             logger.info("Pumps OCR Text:\n" + ocr_text)
#         pumps_list = parse_pumps_table(ocr_text)
#         circ_rates_list = parse_drilling_circ_rates(ocr_text)
#         result = {
#             "Pumps": pumps_list,
#             "DrillingCircRates": circ_rates_list
#         }
#         df_pumps = pd.DataFrame(pumps_list)
#         df_circ = pd.DataFrame(circ_rates_list)
#         if not df_pumps.empty and not df_circ.empty:
#             df = pd.concat([df_pumps, df_circ], axis=0, ignore_index=True)
#         elif not df_pumps.empty:
#             df = df_pumps
#         else:
#             df = df_circ
#         # Return as a dictionary under key "PUMPS"
#         return {"PUMPS": result}, df

# # =============================================================================
# # Time Breakdown Pipeline
# # =============================================================================
# class TimeBreakdownPipeline:
#     @staticmethod
#     def parse_row_text(row_text: str):
#         clean_text = " ".join(row_text.split())
#         # Check if this is a Daily Summary row
#         if "Daily Hrs" in clean_text:
#             pattern = r"Daily Hrs\s+(\S+)\s+Daily NPT Hrs\s*(\S*)\s+Total Job NPT Hours\s+(\S+)"
#             m = re.search(pattern, clean_text, re.IGNORECASE)
#             if m:
#                 logger.info(f"Daily Summary row detected: {clean_text}")
#                 return {"Daily Summary": {
#                     "Daily Hrs": m.group(1),
#                     "Daily NPT Hrs": m.group(2),
#                     "Total Job NPT Hours": m.group(3)
#                 }}
#             else:
#                 logger.warning(f"Daily summary row detected but could not parse: {clean_text}")
#                 return None

#         tokens = clean_text.split()
#         if not tokens or not re.match(r"\d{2}:\d{2}", tokens[0]):
#             logger.debug(f"Skipping header/invalid row: {clean_text}")
#             return None
#         if len(tokens) < 8:
#             logger.warning(f"Row does not have enough tokens: {clean_text}")
#             return None
#         from_time, to_time, hours, depth_start, depth_end = tokens[0:5]
#         header_rest = " ".join(tokens[5:])
#         m = re.search(r"^(?P<phase>.+?)\s+(?P<activity>DR[-]?Drilling)\s+(?P<ops>.*)$", 
#                       header_rest, re.IGNORECASE)
#         if m:
#             phase = m.group("phase")
#             activity = m.group("activity")
#             ops = m.group("ops")
#         else:
#             phase = tokens[5]
#             activity = tokens[6] if len(tokens) > 6 else ""
#             ops = " ".join(tokens[7:]) if len(tokens) > 7 else ""
#         return {
#             "From": from_time,
#             "To": to_time,
#             "Hours": hours,
#             "Depth Start": depth_start,
#             "Depth End": depth_end,
#             "Phase": phase,
#             "Activity": activity,
#             "Operations Description": TimeBreakdownPipeline.parse_operations_description(ops)
#         }

#     @staticmethod
#     def parse_operations_description(ops_text: str):
#         # (Parsing logic remains unchanged; see your provided code.)
#         ops_data = {
#             "Depth": {"From": "", "To": ""},
#             "Performance": {"Feet": "", "FPH": ""},
#             "Rotation_Slide": {"Rotate": "", "Slide": ""},
#             "Rotation_Time": {"Rotate Time": "", "Slide Time": ""},
#             "GPM": "",
#             "MTR RPM": "",
#             "SPP": "",
#             "DIFF": "",
#             "WOB": "",
#             "ROT RPM": "",
#             "ON BTM TRQ": "",
#             "OFF BTM TRQ": "",
#             "GAS": {"Units": "", "Flare": ""},
#             "MW": {"In": "", "Out": ""},
#             "Targets": [],
#             "Observations": []
#         }
#         depth_match = re.search(r"F/\s*([\d,']+)\s*T/\s*([\d,']+)", ops_text, re.IGNORECASE)
#         if depth_match:
#             ops_data["Depth"]["From"] = depth_match.group(1)
#             ops_data["Depth"]["To"] = depth_match.group(2)
#         perf_match = re.search(r"\(([\d,']+)\s*@\s*(\d+)\s*FPH\)", ops_text, re.IGNORECASE)
#         if perf_match:
#             ops_data["Performance"]["Feet"] = perf_match.group(1)
#             ops_data["Performance"]["FPH"] = perf_match.group(2)
#         rs_match = re.search(r"ROTATE\s*([\d.]+%)\s*/\s*SLIDE\s*([\d.]+%)", ops_text, re.IGNORECASE)
#         if rs_match:
#             ops_data["Rotation_Slide"]["Rotate"] = rs_match.group(1)
#             ops_data["Rotation_Slide"]["Slide"] = rs_match.group(2)
#         rt_match = re.search(r"ROTATE\s*TIME\s*([\d.]+%)\s*/\s*SLIDE\s*TIME\s*([\d.]+%)", ops_text, re.IGNORECASE)
#         if rt_match:
#             ops_data["Rotation_Time"]["Rotate Time"] = rt_match.group(1)
#             ops_data["Rotation_Time"]["Slide Time"] = rt_match.group(2)
#         numeric_patterns = {
#             "GPM": r"GPM:\s*(\d+)",
#             "MTR RPM": r"MTR\s*RPM:\s*(\d+)",
#             "SPP": r"SPP:\s*([\d,]+(?:-\d+)?)(?:,|\s|$)",
#             "DIFF": r"DIFF:\s*([\d\-]+)",
#             "WOB": r"WOB:\s*([\d,]+(?:-\d+)?)(?:,|\s|$)",
#             "ROT RPM": r"ROT\s*RPM:\s*([\d,]+(?:-\d+)?)(?:,|\s|$)",
#             "ON BTM TRQ": r"ON\s*BTM\s*TRQ[:;]?\s*([\d\-K]+)",
#             "OFF BTM TRQ": r"OFF\s*BTM\s*TRQ[:;]?\s*([\d\-K]+)"
#         }
#         for key, pattern in numeric_patterns.items():
#             m = re.search(pattern, ops_text, re.IGNORECASE)
#             if m:
#                 ops_data[key] = m.group(1)
#         gas_units = re.search(r"GAS:\s*([\d,]+)\s*UNITS", ops_text, re.IGNORECASE)
#         if gas_units:
#             ops_data["GAS"]["Units"] = gas_units.group(1)
#         flare = re.search(r"(NO\s*FLARE|FLARE\s*ON|FLARE\s*\S+)", ops_text, re.IGNORECASE)
#         if flare:
#             ops_data["GAS"]["Flare"] = flare.group(1)
#         mw_match = re.search(r"MW\s*IN\s*([\d.+]+)\s*PPG\s*/\s*OUT\s*([\d.+]+)\s*PPG", ops_text, re.IGNORECASE)
#         if mw_match:
#             ops_data["MW"]["In"] = mw_match.group(1)
#             ops_data["MW"]["Out"] = mw_match.group(2)
#         segments = re.split(r'(?=\*\*\*)', ops_text)
#         obs_list = []
#         for seg in segments:
#             seg = seg.strip()
#             if not seg:
#                 continue
#             if not seg.startswith('***'):
#                 parts = [p.strip() for p in seg.split('.') if p.strip()]
#                 obs_list.extend(parts)
#             else:
#                 obs_list.append(seg)
#         obs_list = [o.lstrip('* ').strip() for o in obs_list]
#         ops_data["Observations"] = [o for o in obs_list if "TARGET" not in o.upper()]
#         ops_data["Targets"] = [o for o in obs_list if "TARGET" in o.upper()]
#         return ops_data

#     @staticmethod
#     def extract_daily_summary(ocr_text: str):
#         for line in ocr_text.splitlines():
#             if "Daily Hrs" in line:
#                 logger.info(f"Daily Summary found: {line}")
#                 pattern = r"Daily Hrs\s+(\S+)\s+Daily NPT Hrs\s*(\S*)\s+Total Job NPT Hours\s+(\S+)"
#                 m = re.search(pattern, line, re.IGNORECASE)
#                 if m:
#                     return {
#                         "Daily Hrs": m.group(1),
#                         "Daily NPT Hrs": m.group(2),
#                         "Total Job NPT Hours": m.group(3)
#                     }
#         logger.info("No Daily Summary found.")
#         return {}

#     @staticmethod
#     def process(image_path: str, debug: bool = False): #-> (dict, pd.DataFrame):
#         img = ImageUtils.safe_read_image(image_path)
#         proc_img = ImageUtils.preprocess_image(img, debug=debug)
#         rois = ImageUtils.detect_text_regions(proc_img, debug=debug)
#         roi_results = ImageUtils.perform_ocr_on_rois(proc_img, rois, debug=debug)
#         rows = []
#         groups = group_ocr_rows(roi_results, y_threshold=20)
#         for group in groups:
#             group_sorted = sorted(group, key=lambda r: r[0])
#             row_text = " ".join(text for (x, y, w, h, text) in group_sorted)
#             parsed = TimeBreakdownPipeline.parse_row_text(row_text)
#             # Skip adding if this group produced a Daily Summary dictionary.
#             if parsed is not None and "Daily Summary" not in parsed:
#                 rows.append(parsed)
#         if not rows:
#             # Fallback: Use full OCR text line-by-line.
#             full_text = ImageUtils.perform_ocr(proc_img, config="--psm 6")
#             for line in full_text.splitlines():
#                 parsed = TimeBreakdownPipeline.parse_row_text(line)
#                 if parsed is not None and "Daily Summary" not in parsed:
#                     rows.append(parsed)
#         # Independently extract the daily summary from the full OCR text.
#         full_ocr_text = ImageUtils.perform_ocr(proc_img, config="--psm 6")
#         daily_summary = TimeBreakdownPipeline.extract_daily_summary(full_ocr_text)
#         # Return in a uniform structure.
#         return {"TIME BREAKDOWN": rows, "DAILY SUMMARY": daily_summary}, pd.json_normalize(rows)


In [0]:

def safe_read_image_bop(image_path: str):
    """
    Top-level function to read an image using OpenCV.
    """
    local_path = FileUtils.dbfs_to_local_path(image_path)
    logger.info(f"Reading image from: {local_path}")
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"File not found: {local_path}")
    img = cv2.imread(local_path)
    if img is None:
        raise ValueError(f"OpenCV failed to load image: {local_path}")
    return img

def safe_read_image_pil_bop(image_path: str) -> Image.Image:
    local_path = FileUtils.dbfs_to_local_path(image_path)
    logger.info(f"Reading image (PIL) from: {local_path}")
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"File not found: {local_path}")
    return Image.open(local_path)

def perform_ocr_bop(img, config="--psm 6") -> str:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray, config=config)
    return text.strip()

def group_ocr_rows_bop(roi_results, y_threshold=20):
    """
    Group OCR result bounding boxes by their y-coordinate.
    """
    rois_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
    rois_with_center.sort(key=lambda r: r[5])
    groups = []
    current_group = []
    current_center = None
    for (x, y, w, h, text, center) in rois_with_center:
        if current_center is None:
            current_group.append((x, y, w, h, text))
            current_center = center
        elif abs(center - current_center) <= y_threshold:
            current_group.append((x, y, w, h, text))
            current_center = (current_center + center) / 2
        else:
            groups.append(current_group)
            current_group = [(x, y, w, h, text)]
            current_center = center
    if current_group:
        groups.append(current_group)
    return groups

class ImageUtils_bop:
    """Shared image utilities."""
    @staticmethod
    def safe_read_image_bop(image_path: str):
        return safe_read_image_bop(image_path)
    
    @staticmethod
    def safe_read_image_pil_bop(image_path: str) -> Image.Image:
        return safe_read_image_pil_bop(image_path)

    @staticmethod
    def preprocess_image_bop(img, debug: bool = False):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 15, 9
        )
        return thresh

    @staticmethod
    def detect_text_regions_bop(thresh_img, debug: bool = False):
        contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rois = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 30 and h > 15:
                rois.append((x, y, w, h))
        rois.sort(key=lambda b: (b[1], b[0]))
        # if debug:
        #     debug_img = cv2.cvtColor(thresh_img, cv2.COLOR_GRAY2BGR)
        #     for (x, y, w, h) in rois:
        #         cv2.rectangle(debug_img, (x, y), (x+w, y+h), (0,255,0), 2)
        #     plt.figure(figsize=(10, 10))
        #     plt.imshow(debug_img)
        #     plt.title("Detected Text Regions")
        #     plt.axis("off")
        #     plt.show()
        return rois

    @staticmethod
    def perform_ocr_on_rois_bop(img, rois, debug: bool = False):
        results = []
        for (x, y, w, h) in rois:
            roi = img[y:y+h, x:x+w]
            text = pytesseract.image_to_string(roi, config="--psm 6").strip() or "[BLANK]"
            results.append((x, y, w, h, text))
            if debug:
                logger.debug(f"OCR result at ({x}, {y}, {w}, {h}): {text}")
        return results

# =============================================================================
# BOP Pipeline (Reused Shared Functions)
# =============================================================================
class BOPPipeline:
    @staticmethod
    def extract_bop_info_bop(text: str) -> dict:
        patterns = {
            "Last BOP Test Date": r"Last BOP Test Date\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
            "Last BOP Drill": r"Last BOP Drill\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
            "Next BOP Test": r"Next BOP Test\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})"
        }
        result = {}
        for key, regex in patterns.items():
            match = re.search(regex, text, re.IGNORECASE)
            result[key] = match.group(1) if match else ""
        return result

    @staticmethod
    def process_bop(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        logger.info(f"BOPPipeline: Processing image '{image_path}'")
        # Use the top-level safe_read_image defined above.
        img = safe_read_image_bop(image_path)
        ocr_text = perform_ocr_bop(img, config="--psm 6")
        if debug:
            logger.debug(f"BOPPipeline OCR Text:\n{ocr_text}")
        bop_info = BOPPipeline.extract_bop_info_bop(ocr_text)
        for key, value in bop_info.items():
            if value:
                logger.info(f"BOPPipeline: {key} -> {value}")
            else:
                logger.warning(f"BOPPipeline: {key} not found in OCR text.")
        df = pd.DataFrame(list(bop_info.items()), columns=["Key", "Value"])
        logger.info("BOPPipeline: Processing complete.")
        return {"BOP": bop_info}, df



In [0]:
def main(debug: bool = False):
    global logger
    logger = configure_logger(debug=debug, log_file="logs/application.log")
    logger.info("Starting main pipeline execution...")

    # Configuration: image paths for each section.
    image_paths = {
        "DAILY DRILLING REPORT": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png",
        "WELL/JOB INFORMATION": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_2.png",
        "MUD": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_3.png",
        "SURVEY DATA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_4.png",
        "DIR INFO": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_5.png",
        "DRILL BITS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_6.png",
        "CASING": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_7.png",
        "BOP": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png",
        "PERSONNEL": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_9.png",
        "DAILY NUMBERS: OBSERVATION & INTERVENTION": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_10.png",
        "BHA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_11.png",
        "PUMPS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png",
        "COST DATA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_13.png",
        "TIME BREAKDOWN": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_14.png",
        "CONSUMABLES": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_2_section_2.png",
        "BIT INFO": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_6.png"
    }

    # Define pipeline functions (None means not implemented yet)
    pipelines = {
        "DAILY DRILLING REPORT": DailyDrillingReportPipeline.process,
        "WELL/JOB INFORMATION": WellJobInfoPipeline.process,
        "MUD": MudPipeline.process,
        "SURVEY DATA": SurveyDataPipeline.process,
        "DIR INFO": DirInfoPipeline.process,
        "DRILL BITS": DrillBitsPipeline.process,
        "CASING": CasingPipeline.process,
        "BOP": BOPPipeline.process_bop,
        "PERSONNEL": PersonnelPipeline.process,
        "DAILY NUMBERS: OBSERVATION & INTERVENTION": ObsIntPipeline.process,
        "BHA": BHAPipeline.process,
        "PUMPS": PumpsPipeline.process,
        "COST DATA": CostDataPipeline.process,
        "TIME BREAKDOWN": TimeBreakdownPipeline.process,
        "CONSUMABLES": ConsumablesPipeline.process,
        #"BIT INFO": BitInfoPipeline.process
    }

    output_folder = FileUtils.dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
    os.makedirs(output_folder, exist_ok=True)

    aggregated_json = {}
    aggregated_df = pd.DataFrame()

    for section, img_path in image_paths.items():
        func = pipelines.get(section)

        if func is None:
            logger.info(f"Skipping section '{section}' — no pipeline implemented yet.")
            continue

        try:
            logger.info(f"Processing section: {section} from {img_path}...")
            data_json, df = func(img_path, debug=debug)

            # Sanitize section name for file saving
            safe_section = FileUtils.sanitize_section_name(section)

            # Save JSON data
            aggregated_json[section] = data_json.get(section, data_json)
            json_file = os.path.join(output_folder, f"{safe_section}.json")
            with open(json_file, "w") as f:
                json.dump(data_json, f, indent=4)

            # Save CSV data if available
            if df is not None:
                aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
                csv_file = os.path.join(output_folder, f"{safe_section}.csv")
                df.to_csv(csv_file, index=False)

            logger.info(f"Section '{section}' results saved.")
        except Exception as e:
            logger.exception(f"Error processing section '{section}': {e}")

    # Save aggregated outputs
    agg_json_path = os.path.join(output_folder, "aggregated_data.json")
    with open(agg_json_path, "w") as f:
        json.dump(aggregated_json, f, indent=4)

    agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
    aggregated_df.to_csv(agg_csv_path, index=False)

    logger.info(f"Aggregated results saved to {agg_json_path} and {agg_csv_path}.")
    print("----- Aggregated JSON Output -----")
    print(json.dumps(aggregated_json, indent=4))


# Entry point
if __name__ == "__main__":
    main(debug=False)


2025-04-09 21:15:40,823 INFO     Starting main pipeline execution...
INFO:PipelineLogger:Starting main pipeline execution...
2025-04-09 21:15:41,067 INFO     Processing section: DAILY DRILLING REPORT from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png...
INFO:PipelineLogger:Processing section: DAILY DRILLING REPORT from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png...
2025-04-09 21:15:41,069 INFO     Reading image using OpenCV from: /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png
INFO:PipelineLogger:Reading image using OpenCV from: /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png
2025-04-09 21:15:42,334 INFO     Daily Drilling Report OCR extraction complete.
INFO:PipelineLogger:Daily Drilling Report OCR extraction complete.
2025-04-09 21:15:42,387 INFO     Section 'DAILY DRILLING REPORT' results saved.
INFO:PipelineLogger:Section 'DAILY DRILLING REPORT' results saved.
2025-04-09 21:15:42,389 INFO     Processing section: WELL/JOB INF

----- Aggregated JSON Output -----
{
    "DAILY DRILLING REPORT": {
        "Report Date": "7/4/2024",
        "Report Num": "11.",
        "Rig": "Cyclone 39"
    },
    "WELL/JOB INFORMATION": {
        "Well Name": "Ross Fee 4371-31-7-15 MH",
        "Job Name": "Drilling",
        "Supervisor(s)": "CHAD MILLER / ED COOLEY",
        "Field": "XBE",
        "Sec/Twn/Rng": "31, 43N, 71W",
        "Phone": "307-315-1908",
        "AFE #": "240098",
        "API #": "49-005-78911",
        "Email": "cyclone39@aec-denver.com",
        "Contractor": "",
        "Elevation": "4913.5",
        "RKB": "27.5",
        "Spud Date": "6/4/2024",
        "Days from Spud": "7.67",
        "Days on Loc": "34",
        "MD/TVD": "20537 FT/10719 FT",
        "24 Hr Footage": "3068",
        "Present Operations": "DRILLING LATERAL @ 20,537'.",
        "Activity Planned": "DRILL LATERAL SECTION TO PLANNED TD @ ~21,226', PUMP TD SWEEPS & CHC, SOOH & L/D DRILL PIPE."
    },
    "MUD": {
        "Type": "

In [0]:
# #bop
# # import os
# # import re
# # import json
# # import cv2
# # import pytesseract
# # import pandas as pd
# # import logging
# # import matplotlib.pyplot as plt
# # from PIL import Image

# # # =============================================================================
# # # Logging Configuration & Utilities
# # # =============================================================================
# # class ExcludeLogsFilter(logging.Filter):
# #     def filter(self, record):
# #         exclude_keywords = [
# #             "spark.databricks.clusterUsageTags.sparkVersion",
# #             "Answer received",
# #             "Command to send"
# #         ]
# #         return not any(kw in record.getMessage() for kw in exclude_keywords)

# # def configure_logger(debug: bool = False, log_file: str = "logs/application.log") -> logging.Logger:
# #     os.makedirs(os.path.dirname(log_file), exist_ok=True)
# #     logger = logging.getLogger("PipelineLogger")
# #     logger.setLevel(logging.DEBUG if debug else logging.INFO)
# #     for h in logger.handlers[:]:
# #         logger.removeHandler(h)
# #     formatter = logging.Formatter("%(asctime)s %(levelname)-8s %(message)s")
# #     fh = logging.FileHandler(log_file)
# #     fh.setFormatter(formatter)
# #     fh.addFilter(ExcludeLogsFilter())
# #     logger.addHandler(fh)
# #     ch = logging.StreamHandler()
# #     ch.setFormatter(formatter)
# #     ch.addFilter(ExcludeLogsFilter())
# #     logger.addHandler(ch)
# #     return logger

# # logger = configure_logger(debug=False)

# # # =============================================================================
# # # Shared Utility Functions and Classes
# # # =============================================================================
# # class FileUtils:
# #     @staticmethod
# #     def dbfs_to_local_path(dbfs_path: str) -> str:
# #         if dbfs_path.startswith("dbfs:/"):
# #             local_path = os.path.join("/dbfs", dbfs_path[len("dbfs:/"):].lstrip("/"))
# #             logger.debug(f"Converted '{dbfs_path}' to '{local_path}'.")
# #             return local_path
# #         return dbfs_path

# #     @staticmethod
# #     def sanitize_section_name(section: str) -> str:
# #         safe_name = section.lower().replace(" ", "_").replace("/", "_").replace(":", "")
# #         logger.debug(f"Sanitized section name '{section}' to '{safe_name}'.")
# #         return safe_name

# def safe_read_image_bop(image_path: str):
#     """
#     Top-level function to read an image using OpenCV.
#     """
#     local_path = FileUtils.dbfs_to_local_path(image_path)
#     logger.info(f"Reading image from: {local_path}")
#     if not os.path.exists(local_path):
#         raise FileNotFoundError(f"File not found: {local_path}")
#     img = cv2.imread(local_path)
#     if img is None:
#         raise ValueError(f"OpenCV failed to load image: {local_path}")
#     return img

# def safe_read_image_pil_bop(image_path: str) -> Image.Image:
#     local_path = FileUtils.dbfs_to_local_path(image_path)
#     logger.info(f"Reading image (PIL) from: {local_path}")
#     if not os.path.exists(local_path):
#         raise FileNotFoundError(f"File not found: {local_path}")
#     return Image.open(local_path)

# def perform_ocr_bop(img, config="--psm 6") -> str:
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     text = pytesseract.image_to_string(gray, config=config)
#     return text.strip()

# def group_ocr_rows_bop(roi_results, y_threshold=20):
#     """
#     Group OCR result bounding boxes by their y-coordinate.
#     """
#     rois_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
#     rois_with_center.sort(key=lambda r: r[5])
#     groups = []
#     current_group = []
#     current_center = None
#     for (x, y, w, h, text, center) in rois_with_center:
#         if current_center is None:
#             current_group.append((x, y, w, h, text))
#             current_center = center
#         elif abs(center - current_center) <= y_threshold:
#             current_group.append((x, y, w, h, text))
#             current_center = (current_center + center) / 2
#         else:
#             groups.append(current_group)
#             current_group = [(x, y, w, h, text)]
#             current_center = center
#     if current_group:
#         groups.append(current_group)
#     return groups

# class ImageUtils_bop:
#     """Shared image utilities."""
#     @staticmethod
#     def safe_read_image_bop(image_path: str):
#         return safe_read_image_bop(image_path)
    
#     @staticmethod
#     def safe_read_image_pil_bop(image_path: str) -> Image.Image:
#         return safe_read_image_pil_bop(image_path)

#     @staticmethod
#     def preprocess_image_bop(img, debug: bool = False):
#         gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#         thresh = cv2.adaptiveThreshold(
#             gray, 255,
#             cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#             cv2.THRESH_BINARY, 15, 9
#         )
#         return thresh

#     @staticmethod
#     def detect_text_regions_bop(thresh_img, debug: bool = False):
#         contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#         rois = []
#         for cnt in contours:
#             x, y, w, h = cv2.boundingRect(cnt)
#             if w > 30 and h > 15:
#                 rois.append((x, y, w, h))
#         rois.sort(key=lambda b: (b[1], b[0]))
#         # if debug:
#         #     debug_img = cv2.cvtColor(thresh_img, cv2.COLOR_GRAY2BGR)
#         #     for (x, y, w, h) in rois:
#         #         cv2.rectangle(debug_img, (x, y), (x+w, y+h), (0,255,0), 2)
#         #     plt.figure(figsize=(10, 10))
#         #     plt.imshow(debug_img)
#         #     plt.title("Detected Text Regions")
#         #     plt.axis("off")
#         #     plt.show()
#         return rois

#     @staticmethod
#     def perform_ocr_on_rois_bop(img, rois, debug: bool = False):
#         results = []
#         for (x, y, w, h) in rois:
#             roi = img[y:y+h, x:x+w]
#             text = pytesseract.image_to_string(roi, config="--psm 6").strip() or "[BLANK]"
#             results.append((x, y, w, h, text))
#             if debug:
#                 logger.debug(f"OCR result at ({x}, {y}, {w}, {h}): {text}")
#         return results

# # =============================================================================
# # BOP Pipeline (Reused Shared Functions)
# # =============================================================================
# class BOPPipeline:
#     @staticmethod
#     def extract_bop_info_bop(text: str) -> dict:
#         patterns = {
#             "Last BOP Test Date": r"Last BOP Test Date\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
#             "Last BOP Drill": r"Last BOP Drill\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
#             "Next BOP Test": r"Next BOP Test\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})"
#         }
#         result = {}
#         for key, regex in patterns.items():
#             match = re.search(regex, text, re.IGNORECASE)
#             result[key] = match.group(1) if match else ""
#         return result

#     @staticmethod
#     def process_bop(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
#         logger.info(f"BOPPipeline: Processing image '{image_path}'")
#         # Use the top-level safe_read_image defined above.
#         img = safe_read_image_bop(image_path)
#         ocr_text = perform_ocr_bop(img, config="--psm 6")
#         if debug:
#             logger.debug(f"BOPPipeline OCR Text:\n{ocr_text}")
#         bop_info = BOPPipeline.extract_bop_info_bop(ocr_text)
#         for key, value in bop_info.items():
#             if value:
#                 logger.info(f"BOPPipeline: {key} -> {value}")
#             else:
#                 logger.warning(f"BOPPipeline: {key} not found in OCR text.")
#         df = pd.DataFrame(list(bop_info.items()), columns=["Key", "Value"])
#         logger.info("BOPPipeline: Processing complete.")
#         return {"BOP": bop_info}, df



# # =============================================================================
# # Main Aggregation Pipeline (Reusing existing shared utilities)
# # =============================================================================
# def main():
#     debug = False  # Set to True for verbose debugging.
#     global logger
#     logger = configure_logger(debug=debug, log_file="logs/ocr_pipeline.log")
#     logger.info("Starting main pipeline execution...")

#     # Define image paths for each section.
#     # image_paths = {
#     #     "BOP": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png",
#     #     # ... other sections as needed.
#     # }

    
#     # Configuration: image paths for each section.
#     image_paths = {
#         "DAILY DRILLING REPORT": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png",
#         "WELL/JOB INFORMATION": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_2.png",
#         "MUD": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_3.png",
#         "SURVEY DATA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_4.png",
#         "DIR INFO": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_5.png",
#         "DRILL BITS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_6.png",
#         "CASING": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_7.png",
#         "BOP": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png",
#         "PERSONNEL": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_9.png",
#         "DAILY NUMBERS: OBSERVATION & INTERVENTION": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_10.png",
#         "BHA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_11.png",
#         "PUMPS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png",
#         "COST DATA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_13.png",
#         "TIME BREAKDOWN": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_14.png",
#         "CONSUMABLES": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_2_section_2.png",
#         "BIT INFO": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_6.png"
#     }

#     output_folder = FileUtils.dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
#     os.makedirs(output_folder, exist_ok=True)
#     aggregated_json = {}
#     aggregated_df = pd.DataFrame()

#     # # Map section names to pipeline functions.
#     # pipelines = {
#     #     "BOP": BOPPipeline.process_bop,
#     #     # ... add other pipelines here.
#     # }

# # Define pipeline functions (None means not implemented yet)
#     pipelines = {
#         "DAILY DRILLING REPORT": DailyDrillingReportPipeline.process,
#         "WELL/JOB INFORMATION": WellJobInfoPipeline.process,
#         "MUD": MudPipeline.process,
#         "SURVEY DATA": SurveyDataPipeline.process,
#         "DIR INFO": DirInfoPipeline.process,
#         "DRILL BITS": DrillBitsPipeline.process,
#         "CASING": CasingPipeline.process,
#         "BOP": BOPPipeline.process_bop,
#         "PERSONNEL": PersonnelPipeline.process,
#         "DAILY NUMBERS: OBSERVATION & INTERVENTION": ObsIntPipeline.process,
#         "BHA": BHAPipeline.process,
#         "PUMPS": PumpsPipeline.process,
#         "COST DATA": CostDataPipeline.process,
#         "TIME BREAKDOWN": TimeBreakdownPipeline.process,
#         "CONSUMABLES": ConsumablesPipeline.process,
#     }
#     for section, img_path in image_paths.items():
#         func = pipelines.get(section)
#         if func is None:
#             logger.info(f"Skipping section '{section}' — no pipeline implemented yet.")
#             continue

#         try:
#             logger.info(f"Processing section: {section} from {img_path}...")
#             data_json, df = func(img_path, debug=debug)
#             safe_section = FileUtils.sanitize_section_name(section)
#             aggregated_json[section] = data_json.get(section, data_json)
#             json_file = os.path.join(output_folder, f"{safe_section}.json")
#             with open(json_file, "w") as f:
#                 json.dump(data_json, f, indent=4)
#             if df is not None:
#                 aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
#                 csv_file = os.path.join(output_folder, f"{safe_section}.csv")
#                 df.to_csv(csv_file, index=False)
#             logger.info(f"Section '{section}' results saved.")
#         except Exception as e:
#             logger.exception(f"Error processing section '{section}': {e}")

#     # Save aggregated outputs.
#     agg_json_path = os.path.join(output_folder, "aggregated_data.json")
#     with open(agg_json_path, "w") as f:
#         json.dump(aggregated_json, f, indent=4)
#     agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
#     aggregated_df.to_csv(agg_csv_path, index=False)
#     logger.info(f"Aggregated results saved to JSON: {agg_json_path} and CSV: {agg_csv_path}.")
#     print("----- Aggregated JSON Output -----")
#     print(json.dumps(aggregated_json, indent=4))


# if __name__ == "__main__":
#     main()

# # # =============================================================================
# # # Main Aggregation Pipeline
# # # =============================================================================
# # def main():
# #     debug = False  # Change to True for debugging.
# #     global logger
# #     logger = configure_logger(debug=debug, log_file="logs/ocr_pipeline.log")
# #     logger.info("Starting main pipeline execution...")

# #     image_paths = {
# #         "BOP": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png"
# #         # Add other sections as needed.
# #     }

# #     output_folder = FileUtils.dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
# #     os.makedirs(output_folder, exist_ok=True)
# #     aggregated_json = {}
# #     aggregated_df = pd.DataFrame()

# #     pipelines = {
# #         "BOP": BOPPipeline.process_bop
# #         # Map other sections to their process functions.
# #     }

# #     for section, img_path in image_paths.items():
# #         func = pipelines.get(section)
# #         if func is None:
# #             logger.info(f"Skipping section '{section}' — no pipeline implemented yet.")
# #             continue

# #         try:
# #             logger.info(f"Processing section: {section} from {img_path}...")
# #             data_json, df = func(img_path, debug=debug)
# #             safe_section = FileUtils.sanitize_section_name(section)
# #             aggregated_json[section] = data_json.get(section, data_json)
# #             json_file = os.path.join(output_folder, f"{safe_section}.json")
# #             with open(json_file, "w") as f:
# #                 json.dump(data_json, f, indent=4)
# #             if df is not None:
# #                 aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
# #                 csv_file = os.path.join(output_folder, f"{safe_section}.csv")
# #                 df.to_csv(csv_file, index=False)
# #             logger.info(f"Section '{section}' results saved.")
# #         except Exception as e:
# #             logger.exception(f"Error processing section '{section}': {e}")

# #     agg_json_path = os.path.join(output_folder, "aggregated_data.json")
# #     with open(agg_json_path, "w") as f:
# #         json.dump(aggregated_json, f, indent=4)
# #     agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
# #     aggregated_df.to_csv(agg_csv_path, index=False)
# #     logger.info(f"Aggregated results saved to JSON: {agg_json_path} and CSV: {agg_csv_path}.")
# #     print("----- Aggregated JSON Output -----")
# #     print(json.dumps(aggregated_json, indent=4))


# # if __name__ == "__main__":
# #     main()




In [0]:
# # =============================================================================
# # BOP Pipeline (Reused Shared Utilities with BOP-specific function names)
# # =============================================================================
# class BOPPipeline:
#     """
#     Pipeline for processing the BOP section.
#     Extracts date fields from the BOP image OCR text.
#     Uses shared utilities for file handling, image reading, and OCR.
#     """

#     @staticmethod
#     def extract_bop_info_bop(text: str) -> dict:
#         """
#         BOP-specific extraction logic. This function reuses your regex patterns 
#         to extract date fields from OCR text.
#         """
#         patterns = {
#             "Last BOP Test Date": r"Last BOP Test Date\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
#             "Last BOP Drill": r"Last BOP Drill\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
#             "Next BOP Test": r"Next BOP Test\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})"
#         }
#         result = {}
#         for key, regex in patterns.items():
#             match = re.search(regex, text, re.IGNORECASE)
#             result[key] = match.group(1) if match else ""
#         return result

#     @staticmethod
#     def process_bop(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
#         """
#         Process the BOP section using shared functions.
#         - Reads the image via safe_read_image.
#         - Performs OCR via perform_ocr.
#         - Extracts BOP information via extract_bop_info_bop.
#         Returns a dictionary with key "BOP" and a DataFrame.
#         """
#         logger.info(f"BOPPipeline: Processing image '{image_path}'")
#         # Reuse shared image reading.
#         img = safe_read_image(image_path)
#         ocr_text = perform_ocr(img, config="--psm 6")
#         if debug:
#             logger.debug(f"BOPPipeline OCR Text:\n{ocr_text}")
#         bop_info = BOPPipeline.extract_bop_info_bop(ocr_text)
#         # Log each extracted field.
#         for key, value in bop_info.items():
#             if value:
#                 logger.info(f"BOPPipeline: {key} -> {value}")
#             else:
#                 logger.warning(f"BOPPipeline: {key} not found in OCR text.")
#         df = pd.DataFrame(list(bop_info.items()), columns=["Key", "Value"])
#         logger.info("BOPPipeline: Processing complete.")
#         # Return a uniform dictionary under key "BOP"
#         return {"BOP": bop_info}, df


# # =============================================================================
# # Main Aggregation Pipeline (Reusing existing shared utilities)
# # =============================================================================
# def main():
#     debug = False  # Set to True for verbose debugging.
#     global logger
#     logger = configure_logger(debug=debug, log_file="logs/ocr_pipeline.log")
#     logger.info("Starting main pipeline execution...")

#     # Define image paths for each section.
#     # image_paths = {
#     #     "BOP": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png",
#     #     # ... other sections as needed.
#     # }

    
#     # Configuration: image paths for each section.
#     image_paths = {
#         "DAILY DRILLING REPORT": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png",
#         "WELL/JOB INFORMATION": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_2.png",
#         "MUD": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_3.png",
#         "SURVEY DATA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_4.png",
#         "DIR INFO": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_5.png",
#         "DRILL BITS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_6.png",
#         "CASING": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_7.png",
#         "BOP": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png",
#         "PERSONNEL": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_9.png",
#         "DAILY NUMBERS: OBSERVATION & INTERVENTION": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_10.png",
#         "BHA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_11.png",
#         "PUMPS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png",
#         "COST DATA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_13.png",
#         "TIME BREAKDOWN": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_14.png",
#         "CONSUMABLES": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_2_section_2.png",
#         "BIT INFO": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_6.png"
#     }

#     output_folder = FileUtils.dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
#     os.makedirs(output_folder, exist_ok=True)
#     aggregated_json = {}
#     aggregated_df = pd.DataFrame()

#     # # Map section names to pipeline functions.
#     # pipelines = {
#     #     "BOP": BOPPipeline.process_bop,
#     #     # ... add other pipelines here.
#     # }

# # Define pipeline functions (None means not implemented yet)
#     pipelines = {
#         "DAILY DRILLING REPORT": DailyDrillingReportPipeline.process,
#         "WELL/JOB INFORMATION": WellJobInfoPipeline.process,
#         "MUD": MudPipeline.process,
#         "SURVEY DATA": SurveyDataPipeline.process,
#         "DIR INFO": DirInfoPipeline.process,
#         "DRILL BITS": DrillBitsPipeline.process,
#         "CASING": CasingPipeline.process,
#         "BOP": BOPPipeline.process_bop,
#         "PERSONNEL": PersonnelPipeline.process,
#         "DAILY NUMBERS: OBSERVATION & INTERVENTION": ObsIntPipeline.process,
#         "BHA": BHAPipeline.process,
#         "PUMPS": PumpsPipeline.process,
#         "COST DATA": CostDataPipeline.process,
#         "TIME BREAKDOWN": TimeBreakdownPipeline.process,
#         "CONSUMABLES": ConsumablesPipeline.process,
#     }
#     for section, img_path in image_paths.items():
#         func = pipelines.get(section)
#         if func is None:
#             logger.info(f"Skipping section '{section}' — no pipeline implemented yet.")
#             continue

#         try:
#             logger.info(f"Processing section: {section} from {img_path}...")
#             data_json, df = func(img_path, debug=debug)
#             safe_section = FileUtils.sanitize_section_name(section)
#             aggregated_json[section] = data_json.get(section, data_json)
#             json_file = os.path.join(output_folder, f"{safe_section}.json")
#             with open(json_file, "w") as f:
#                 json.dump(data_json, f, indent=4)
#             if df is not None:
#                 aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
#                 csv_file = os.path.join(output_folder, f"{safe_section}.csv")
#                 df.to_csv(csv_file, index=False)
#             logger.info(f"Section '{section}' results saved.")
#         except Exception as e:
#             logger.exception(f"Error processing section '{section}': {e}")

#     # Save aggregated outputs.
#     agg_json_path = os.path.join(output_folder, "aggregated_data.json")
#     with open(agg_json_path, "w") as f:
#         json.dump(aggregated_json, f, indent=4)
#     agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
#     aggregated_df.to_csv(agg_csv_path, index=False)
#     logger.info(f"Aggregated results saved to JSON: {agg_json_path} and CSV: {agg_csv_path}.")
#     print("----- Aggregated JSON Output -----")
#     print(json.dumps(aggregated_json, indent=4))


# if __name__ == "__main__":
#     main()


In [0]:
#!/usr/bin/env python3
"""
Integrated OCR Extraction Pipeline for BOP, Pumps, and Time Breakdown.

This module reuses shared functions to read images, convert DBFS paths,
perform OCR, and group results. Each pipeline returns a uniform dictionary:
 - BOPPipeline returns {"BOP": { ... }}
 - PumpsPipeline returns {"PUMPS": {"Pumps": [...], "DrillingCircRates": [...]}}
 - TimeBreakdownPipeline returns {"TIME BREAKDOWN": [...], "DAILY SUMMARY": {...}}
"""

import os
import re
import json
import cv2
import pytesseract
import pandas as pd
import logging
import matplotlib.pyplot as plt
from PIL import Image

# =============================================================================
# Logging Configuration & Utilities
# =============================================================================
class ExcludeLogsFilter(logging.Filter):
    def filter(self, record):
        unwanted = [
            "spark.databricks.clusterUsageTags.sparkVersion",
            "Answer received",
            "Command to send"
        ]
        return not any(kw in record.getMessage() for kw in unwanted)

def configure_logger(debug: bool = False, log_file: str = "logs/ocr_pipeline.log") -> logging.Logger:
    os.makedirs(os.path.dirname(log_file), exist_ok=True)
    logger = logging.getLogger("PipelineLogger")
    logger.setLevel(logging.DEBUG if debug else logging.INFO)
    for h in logger.handlers[:]:
        logger.removeHandler(h)
    formatter = logging.Formatter("%(asctime)s %(levelname)-8s %(message)s")
    fh = logging.FileHandler(log_file)
    fh.setFormatter(formatter)
    fh.addFilter(ExcludeLogsFilter())
    logger.addHandler(fh)
    # Only add a stream handler in debug mode.
    if debug:
        ch = logging.StreamHandler()
        ch.setFormatter(formatter)
        ch.addFilter(ExcludeLogsFilter())
        logger.addHandler(ch)
    return logger

logger = configure_logger(debug=False)

# =============================================================================
# Shared Utilities and Functions
# =============================================================================
class FileUtils:
    @staticmethod
    def dbfs_to_local_path(dbfs_path: str) -> str:
        if dbfs_path.startswith("dbfs:/"):
            local_path = os.path.join("/dbfs", dbfs_path[len("dbfs:/"):].lstrip("/"))
            logger.debug(f"Converted '{dbfs_path}' to '{local_path}'.")
            return local_path
        return dbfs_path

    @staticmethod
    def sanitize_section_name(section: str) -> str:
        safe_name = section.lower().replace(" ", "_").replace("/", "_").replace(":", "")
        logger.debug(f"Sanitized section name '{section}' to '{safe_name}'.")
        return safe_name

def safe_read_image(image_path: str):
    local_path = FileUtils.dbfs_to_local_path(image_path)
    logger.info(f"Reading image from: {local_path}")
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"File not found: {local_path}")
    img = cv2.imread(local_path)
    if img is None:
        raise ValueError(f"OpenCV failed to load image: {local_path}")
    return img

def safe_read_image_pil(image_path: str) -> Image.Image:
    local_path = FileUtils.dbfs_to_local_path(image_path)
    logger.info(f"Reading image with PIL from: {local_path}")
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"File not found: {local_path}")
    return Image.open(local_path)

def perform_ocr(img, config="--psm 6") -> str:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray, config=config)
    return text.strip()

def group_ocr_rows(roi_results, y_threshold=20):
    """
    Groups OCR result bounding boxes (tuples of (x, y, w, h, text)) by their y-coordinate.
    """
    rois_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
    rois_with_center.sort(key=lambda r: r[5])
    groups = []
    current_group = []
    current_center = None
    for (x, y, w, h, text, center) in rois_with_center:
        if current_center is None:
            current_group.append((x, y, w, h, text))
            current_center = center
        elif abs(center - current_center) <= y_threshold:
            current_group.append((x, y, w, h, text))
            current_center = (current_center + center) / 2
        else:
            groups.append(current_group)
            current_group = [(x, y, w, h, text)]
            current_center = center
    if current_group:
        groups.append(current_group)
    return groups

class ImageUtils:
    @staticmethod
    def safe_read_image(image_path: str):
        return safe_read_image(image_path)
    
    @staticmethod
    def safe_read_image_pil(image_path: str) -> Image.Image:
        return safe_read_image_pil(image_path)

    @staticmethod
    def preprocess_image(img, debug: bool = False):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 15, 9)
        # )
        # if debug:
        #     plt.figure(figsize=(8, 8))
        #     plt.imshow(thresh, cmap="gray")
        #     plt.title("Thresholded Image")
        #     plt.axis("off")
        #     plt.show()
        return thresh

    @staticmethod
    def detect_text_regions(thresh_img, debug: bool = False):
        contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rois = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 30 and h > 15:
                rois.append((x, y, w, h))
        rois.sort(key=lambda b: (b[1], b[0]))
        if debug:
            debug_img = cv2.cvtColor(thresh_img, cv2.COLOR_GRAY2BGR)
            for (x, y, w, h) in rois:
                cv2.rectangle(debug_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
            plt.figure(figsize=(10, 10))
            plt.imshow(debug_img)
            plt.title("Detected Text Regions")
            plt.axis("off")
            plt.show()
        return rois

    @staticmethod
    def perform_ocr_on_rois(img, rois, debug: bool = False):
        results = []
        for (x, y, w, h) in rois:
            roi = img[y:y+h, x:x+w]
            text = pytesseract.image_to_string(roi, config="--psm 6").strip() or "[BLANK]"
            results.append((x, y, w, h, text))
            if debug:
                logger.debug(f"OCR result at ({x}, {y}, {w}, {h}): {text}")
        return results

# =============================================================================
# BOP Pipeline
# =============================================================================
class BOPPipeline:
    @staticmethod
    def extract_bop_info_bop(text: str) -> dict:
        patterns = {
            "Last BOP Test Date": r"Last BOP Test Date\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
            "Last BOP Drill": r"Last BOP Drill\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
            "Next BOP Test": r"Next BOP Test\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})"
        }
        result = {}
        for key, regex in patterns.items():
            match = re.search(regex, text, re.IGNORECASE)
            result[key] = match.group(1) if match else ""
        return result

    @staticmethod
    def process_bop(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        logger.info(f"BOPPipeline: Processing image '{image_path}'")
        img = safe_read_image(image_path)
        ocr_text = perform_ocr(img, config="--psm 6")
        if debug:
            logger.debug(f"BOPPipeline OCR Text:\n{ocr_text}")
        bop_info = BOPPipeline.extract_bop_info_bop(ocr_text)
        for key, value in bop_info.items():
            if value:
                logger.info(f"BOPPipeline: {key} -> {value}")
            else:
                logger.warning(f"BOPPipeline: {key} not found in OCR text.")
        df = pd.DataFrame(list(bop_info.items()), columns=["Key", "Value"])
        logger.info("BOPPipeline: Processing complete.")
        return {"BOP": bop_info}, df

# =============================================================================
# Pumps Pipeline
# =============================================================================
def parse_pumps_table(ocr_text: str) -> list:
    pump_pattern = re.compile(
        r"^(\d+)?\s*"
        r"(BOMCO)\s+(TRIPLEX)\s+"
        r"(\d+)?\s*"
        r"(\d+)\s+"
        r"([\d.]+)\s+"
        r"([\d.]+)\s+"
        r"(\d+)\s+"
        r"(\d+)\s+"
        r"(\d+)\s+"
        r"(\d+)\s*$",
        re.IGNORECASE
    )
    pumps = []
    for line in ocr_text.splitlines():
        line = line.strip()
        match = pump_pattern.match(line)
        if match:
            (number, model, pump_type, hhp, efficiency, stroke,
             liner, p_rating, p_limit, spm_rating, spm_limit) = match.groups()
            pumps.append({
                "Number": number if number else "",
                "Model": model,
                "Type": pump_type,
                "HHP": hhp if hhp else "",
                "Efficiency": efficiency,
                "Stroke(in)": stroke,
                "Liner(in)": liner,
                "P-Rating(psi)": p_rating,
                "P-Limit(psi)": p_limit,
                "SPM Rating": spm_rating,
                "SPM Limit": spm_limit
            })
    return pumps

def parse_drilling_circ_rates(ocr_text: str) -> list:
    circ_rates = []
    segments = re.split(r"(?=Drilling/Circ Rate \d+)", ocr_text)
    pattern = re.compile(
        r"Drilling/Circ Rate\s+(\d+).*?"
        r"(\d+)\s+PS[!I].*?"
        r"@\s*(\d+).*?"
        r"([\d.]+)\s+Gal/Stoke.*?"
        r"([\d.]+)\s+GPM.*?"
        r"([\d.]+)\s+BPM.*?"
        r"([\d.]+)\s+DC.*?"
        r"([\d.]+)\s+DP",
        re.IGNORECASE | re.DOTALL
    )
    for seg in segments:
        seg = seg.strip()
        if not seg.startswith("Drilling/Circ Rate"):
            continue
        seg_clean = " ".join(seg.splitlines())
        match = pattern.search(seg_clean)
        if match:
            rate_id, pressure, spm, gal_stroke, gpm, bpm, dc, dp = match.groups()
            circ_rates.append({
                "RateID": rate_id,
                "Pressure(PSI)": pressure,
                "SPM": spm,
                "Gal/Stoke": gal_stroke,
                "GPM": gpm,
                "BPM": bpm,
                "DC": dc,
                "DP": dp
            })
        else:
            logger.warning(f"PumpsPipeline: No match in segment:\n{seg_clean}")
    return circ_rates

class PumpsPipeline:
    @staticmethod
    def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        pil_img = safe_read_image_pil(image_path)
        ocr_text = perform_ocr(pil_img, config="--psm 6")
        if debug:
            logger.info("Pumps OCR Text:\n" + ocr_text)
        pumps_list = parse_pumps_table(ocr_text)
        circ_rates_list = parse_drilling_circ_rates(ocr_text)
        result = {"Pumps": pumps_list, "DrillingCircRates": circ_rates_list}
        df_pumps = pd.DataFrame(pumps_list)
        df_circ = pd.DataFrame(circ_rates_list)
        if not df_pumps.empty and not df_circ.empty:
            df = pd.concat([df_pumps, df_circ], axis=0, ignore_index=True)
        elif not df_pumps.empty:
            df = df_pumps
        else:
            df = df_circ
        logger.info("PumpsPipeline: Processing complete.")
        return {"PUMPS": result}, df

# =============================================================================
# Time Breakdown Pipeline
# =============================================================================
class TimeBreakdownPipeline:
    @staticmethod
    def parse_row_text(row_text: str):
        clean_text = " ".join(row_text.split())
        # Check for Daily Summary row
        if "Daily Hrs" in clean_text:
            pattern = r"Daily Hrs\s+(\S+)\s+Daily NPT Hrs\s*(\S*)\s+Total Job NPT Hours\s+(\S+)"
            m = re.search(pattern, clean_text, re.IGNORECASE)
            if m:
                logger.info(f"Daily Summary row detected: {clean_text}")
                return {"Daily Summary": {
                    "Daily Hrs": m.group(1),
                    "Daily NPT Hrs": m.group(2),
                    "Total Job NPT Hours": m.group(3)
                }}
            else:
                logger.warning(f"Daily summary row detected but could not parse: {clean_text}")
                return None

        tokens = clean_text.split()
        if not tokens or not re.match(r"\d{2}:\d{2}", tokens[0]):
            logger.debug(f"Skipping header/invalid row: {clean_text}")
            return None
        if len(tokens) < 8:
            logger.warning(f"Row does not have enough tokens: {clean_text}")
            return None
        from_time, to_time, hours, depth_start, depth_end = tokens[0:5]
        header_rest = " ".join(tokens[5:])
        m = re.search(r"^(?P<phase>.+?)\s+(?P<activity>DR[-]?Drilling)\s+(?P<ops>.*)$",
                      header_rest, re.IGNORECASE)
        if m:
            phase = m.group("phase")
            activity = m.group("activity")
            ops = m.group("ops")
        else:
            phase = tokens[5]
            activity = tokens[6] if len(tokens) > 6 else ""
            ops = " ".join(tokens[7:]) if len(tokens) > 7 else ""
        return {
            "From": from_time,
            "To": to_time,
            "Hours": hours,
            "Depth Start": depth_start,
            "Depth End": depth_end,
            "Phase": phase,
            "Activity": activity,
            "Operations Description": TimeBreakdownPipeline.parse_operations_description(ops)
        }

    @staticmethod
    def parse_operations_description(ops_text: str):
        # (Using your existing parsing logic; adjust as needed.)
        ops_data = {
            "Depth": {"From": "", "To": ""},
            "Performance": {"Feet": "", "FPH": ""},
            "Rotation_Slide": {"Rotate": "", "Slide": ""},
            "Rotation_Time": {"Rotate Time": "", "Slide Time": ""},
            "GPM": "",
            "MTR RPM": "",
            "SPP": "",
            "DIFF": "",
            "WOB": "",
            "ROT RPM": "",
            "ON BTM TRQ": "",
            "OFF BTM TRQ": "",
            "GAS": {"Units": "", "Flare": ""},
            "MW": {"In": "", "Out": ""},
            "Targets": [],
            "Observations": []
        }
        # (Insert your regex-based parsing here.)
        return ops_data

    @staticmethod
    def extract_daily_summary(ocr_text: str):
        for line in ocr_text.splitlines():
            if "Daily Hrs" in line:
                logger.info(f"Daily Summary found: {line}")
                pattern = r"Daily Hrs\s+(\S+)\s+Daily NPT Hrs\s*(\S*)\s+Total Job NPT Hours\s+(\S+)"
                m = re.search(pattern, line, re.IGNORECASE)
                if m:
                    return {
                        "Daily Hrs": m.group(1),
                        "Daily NPT Hrs": m.group(2),
                        "Total Job NPT Hours": m.group(3)
                    }
        logger.info("No Daily Summary found.")
        return {}

    @staticmethod
    def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        img = safe_read_image(image_path)
        proc_img = preprocess_image(img, debug=debug)
        rois = detect_text_regions(proc_img, debug=debug)
        roi_results = perform_ocr_on_rois(proc_img, rois, debug=debug)
        rows = []
        groups = group_ocr_rows(roi_results, y_threshold=20)
        for group in groups:
            group_sorted = sorted(group, key=lambda r: r[0])
            row_text = " ".join(text for (x, y, w, h, text) in group_sorted)
            parsed = TimeBreakdownPipeline.parse_row_text(row_text)
            if parsed is not None and "Daily Summary" not in parsed:
                rows.append(parsed)
        if not rows:
            full_text = perform_ocr(proc_img, config="--psm 6")
            for line in full_text.splitlines():
                parsed = TimeBreakdownPipeline.parse_row_text(line)
                if parsed is not None and "Daily Summary" not in parsed:
                    rows.append(parsed)
        full_ocr_text = perform_ocr(proc_img, config="--psm 6")
        daily_summary = TimeBreakdownPipeline.extract_daily_summary(full_ocr_text)
        logger.info("TimeBreakdownPipeline: Processing complete.")
        return {"TIME BREAKDOWN": rows, "DAILY SUMMARY": daily_summary}, pd.json_normalize(rows)

# =============================================================================
# Main Aggregation Pipeline
# =============================================================================
def main(debug: bool = False):
    global logger
    logger = configure_logger(debug=debug, log_file="logs/ocr_pipeline.log")
    logger.info("Starting main pipeline execution...")

    image_paths = {
        "BOP": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png",
        "PUMPS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png",
        "TIME BREAKDOWN": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_14.png"
    }

    output_folder = FileUtils.dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
    os.makedirs(output_folder, exist_ok=True)
    aggregated_json = {}
    aggregated_df = pd.DataFrame()

    pipelines = {
        "BOP": BOPPipeline.process_bop,
        "PUMPS": PumpsPipeline.process,
        "TIME BREAKDOWN": TimeBreakdownPipeline.process
    }

    for section, img_path in image_paths.items():
        func = pipelines.get(section)
        if func is None:
            logger.info(f"Skipping section '{section}' — no pipeline implemented.")
            continue
        try:
            logger.info(f"Processing section: {section} from {img_path}...")
            data_json, df = func(img_path, debug=debug)
            safe_section = FileUtils.sanitize_section_name(section)
            aggregated_json[section] = data_json.get(section, data_json)
            json_file = os.path.join(output_folder, f"{safe_section}.json")
            with open(json_file, "w") as f:
                json.dump(data_json, f, indent=4)
            if df is not None:
                aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
                csv_file = os.path.join(output_folder, f"{safe_section}.csv")
                df.to_csv(csv_file, index=False)
            logger.info(f"Section '{section}' results saved.")
        except Exception as e:
            logger.exception(f"Error processing section '{section}': {e}")

    agg_json_path = os.path.join(output_folder, "aggregated_data.json")
    with open(agg_json_path, "w") as f:
        json.dump(aggregated_json, f, indent=4)
    agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
    aggregated_df.to_csv(agg_csv_path, index=False)
    logger.info(f"Aggregated results saved to JSON: {agg_json_path} and CSV: {agg_csv_path}.")
    print("----- Aggregated JSON Output -----")
    print(json.dumps(aggregated_json, indent=4))

if __name__ == "__main__":
    main(debug=False)


INFO:PipelineLogger:Starting main pipeline execution...
INFO:PipelineLogger:Processing section: BOP from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png...
INFO:PipelineLogger:BOPPipeline: Processing image 'dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png'
INFO:PipelineLogger:Reading image from: /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png
INFO:PipelineLogger:BOPPipeline: Last BOP Test Date -> 6/30/24
INFO:PipelineLogger:BOPPipeline: Last BOP Drill -> 7/3/2024
INFO:PipelineLogger:BOPPipeline: Next BOP Test -> 7/25/24
INFO:PipelineLogger:BOPPipeline: Processing complete.
INFO:PipelineLogger:Section 'BOP' results saved.
INFO:PipelineLogger:Processing section: PUMPS from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png...
INFO:PipelineLogger:Reading image with PIL from: /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png
ERROR:PipelineLogger:Error processing section 'PUMPS': OpenCV(4.11.0) :-1: error: (-5:Bad argument) in fun

----- Aggregated JSON Output -----
{
    "BOP": {
        "Last BOP Test Date": "6/30/24",
        "Last BOP Drill": "7/3/2024",
        "Next BOP Test": "7/25/24"
    }
}


In [0]:
#bop
# import os
# import re
# import json
# import cv2
# import pytesseract
# import pandas as pd
# import logging
# from PIL import Image

# # -----------------------------------------------------------------------------
# # Logging Configuration & Utilities
# # -----------------------------------------------------------------------------
# class ExcludeLogsFilter(logging.Filter):
#     """
#     Filters out unwanted log messages.
#     """
#     def filter(self, record):
#         unwanted = [
#             "spark.databricks.clusterUsageTags.sparkVersion",
#             "Answer received",
#             "Command to send"
#         ]
#         return not any(kw in record.getMessage() for kw in unwanted)

# def configure_logger(debug: bool = False, log_file: str = "logs/ocr_pipeline.log") -> logging.Logger:
#     """
#     Configures and returns a logger.
#     In production (debug=False) only essential logs are emitted (and saved to file).
#     """
#     os.makedirs(os.path.dirname(log_file), exist_ok=True)
#     logger = logging.getLogger("WellJobInfoExtractor")
#     logger.setLevel(logging.DEBUG if debug else logging.INFO)
#     # Remove existing handlers
#     for h in logger.handlers[:]:
#         logger.removeHandler(h)
#     formatter = logging.Formatter("%(asctime)s %(levelname)-8s %(message)s")
#     fh = logging.FileHandler(log_file)
#     fh.setFormatter(formatter)
#     fh.addFilter(ExcludeLogsFilter())
#     logger.addHandler(fh)
#     if debug:
#         ch = logging.StreamHandler()
#         ch.setFormatter(formatter)
#         ch.addFilter(ExcludeLogsFilter())
#         logger.addHandler(ch)
#     return logger

# # Global logger instance
# logger = configure_logger(debug=False)

# # -----------------------------------------------------------------------------
# # Shared Utilities
# # -----------------------------------------------------------------------------
# class FileUtils:
#     @staticmethod
#     def dbfs_to_local_path(dbfs_path: str) -> str:
#         """
#         Converts a DBFS path (e.g., "dbfs:/mnt/xxx") to a local path ("/dbfs/mnt/xxx").
#         """
#         if dbfs_path.startswith("dbfs:/"):
#             local_path = os.path.join("/dbfs", dbfs_path[len("dbfs:/"):].lstrip("/"))
#             logger.debug(f"Converted '{dbfs_path}' to '{local_path}'")
#             return local_path
#         return dbfs_path

# def safe_read_image(image_path: str):
#     """
#     Reads an image from a DBFS or local path using OpenCV.
#     """
#     local_path = FileUtils.dbfs_to_local_path(image_path)
#     logger.info(f"Reading image from: {local_path}")
#     if not os.path.exists(local_path):
#         raise FileNotFoundError(f"File not found: {local_path}")
#     img = cv2.imread(local_path)
#     if img is None:
#         raise ValueError(f"OpenCV failed to load image: {local_path}")
#     return img

# def safe_read_image_pil(image_path: str) -> Image.Image:
#     """
#     Reads an image from a DBFS or local path using PIL.
#     """
#     local_path = FileUtils.dbfs_to_local_path(image_path)
#     logger.info(f"Reading image with PIL from: {local_path}")
#     if not os.path.exists(local_path):
#         raise FileNotFoundError(f"File not found: {local_path}")
#     img = Image.open(local_path)
#     return img

# def perform_ocr(img, config="--psm 6") -> str:
#     """
#     Performs OCR on the image using pytesseract.
#     """
#     # Convert to grayscale (if using OpenCV) for better OCR results.
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     text = pytesseract.image_to_string(gray, config=config)
#     return text

# # -----------------------------------------------------------------------------
# # Output Helper
# # -----------------------------------------------------------------------------
# def save_output(section: str, data_json: dict, df: pd.DataFrame, output_folder: str):
#     """
#     Saves JSON and CSV outputs for a given section.
#     """
#     section_key = section.lower().replace(" ", "_")
#     json_path = os.path.join(output_folder, f"{section_key}.json")
#     csv_path = os.path.join(output_folder, f"{section_key}.csv")
#     with open(json_path, "w") as f:
#         json.dump(data_json, f, indent=4)
#     df.to_csv(csv_path, index=False)
#     logger.info(f"{section} output saved to JSON: {json_path} and CSV: {csv_path}")

# # -----------------------------------------------------------------------------
# # BOP Pipeline Class
# # -----------------------------------------------------------------------------
# class BOPPipeline:
#     """
#     Pipeline for processing the BOP section.
#     Extracts date fields from the OCR text.
#     """
#     @staticmethod
#     def extract_bop_info(text: str) -> dict:
#         pattern = {
#             "Last BOP Test Date": r"Last BOP Test Date\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
#             "Last BOP Drill": r"Last BOP Drill\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
#             "Next BOP Test": r"Next BOP Test\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})"
#         }
#         result = {}
#         for key, regex in pattern.items():
#             match = re.search(regex, text, re.IGNORECASE)
#             result[key] = match.group(1) if match else ""
#         return result

#     @staticmethod
#     def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
#         logger.info(f"BOPPipeline: Processing image '{image_path}'")
#         img = safe_read_image(image_path)
#         ocr_text = perform_ocr(img, config="--psm 6")
#         logger.debug(f"BOPPipeline: OCR Text:\n{ocr_text}")
#         bop_info = BOPPipeline.extract_bop_info(ocr_text)
#         for key, value in bop_info.items():
#             if value:
#                 logger.info(f"BOPPipeline: {key} -> {value}")
#             else:
#                 logger.warning(f"BOPPipeline: {key} not found in OCR text.")
#         df = pd.DataFrame(list(bop_info.items()), columns=["Key", "Value"])
#         logger.info("BOPPipeline: Processing complete.")
#         return {"BOP": bop_info}, df


# # -----------------------------------------------------------------------------
# # Main Aggregation Pipeline
# # -----------------------------------------------------------------------------
# def main():
#     debug = False  # Set to True for verbose debugging.
#     global logger
#     logger = configure_logger(debug=debug, log_file="logs/ocr_pipeline.log")
#     logger.info("Main pipeline started for Pumps and BOP sections.")

#     image_paths = {
#         "BOP":   "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png"
#     }
#     output_folder = FileUtils.dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
#     os.makedirs(output_folder, exist_ok=True)
#     aggregated_json = {}
#     aggregated_df = pd.DataFrame()

#     processes = [
#         ("BOP", BOPPipeline.process, image_paths.get("BOP"))
#     ]
    
#     for section, func, img_path in processes:
#         try:
#             logger.info(f"Main: Processing section '{section}'")
#             data_json, df = func(img_path, debug)
#             aggregated_json[section] = data_json.get(section, data_json)
#             aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
#             logger.info(f"Main: {section} output:\n{json.dumps(data_json, indent=4)}")
#             save_output(section, data_json, df, output_folder)
#         except Exception as e:
#             logger.error(f"Main: Error processing section '{section}': {e}")
    
#     # Save aggregated results.
#     agg_json_path = os.path.join(output_folder, "aggregated_data.json")
#     with open(agg_json_path, "w") as f:
#         json.dump(aggregated_json, f, indent=4)
#     agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
#     aggregated_df.to_csv(agg_csv_path, index=False)
#     logger.info(f"Main: Aggregated results saved to JSON: {agg_json_path}")
#     print("----- Aggregated JSON Output -----")
#     print(json.dumps(aggregated_json, indent=4))

# if __name__ == "__main__":
#     main()
